# Лекция 6: "Самый Главный и Лысый". От письма к Архитектуре.

## 1. Входящие данные

На прошлом занятии вы получили **Письмо Счастья** от Ректората.
Давайте перечитаем его уже не как "программисты", которые хотят писать код, а как **Системные Аналитики**, которые хотят сдать проект и получить деньги.

> **Важно:** В письме спрятаны мины. Если наступите на них сейчас — проект взорвется через месяц.

*(Ниже текст письма для освежения памяти)*
---
**От:** Ректорат ВУЗа (noreply@university.edu)
**Тема:** Коммерческое предложение

**ТЗ (Выжимка):**
1.  **Формат:** CSV (загрузка в чаты).
2.  **Функции:** Фильтр по группе, Расписание по запросу.
3.  **Уведомления:** За 1 час. Только нужным людям.
4.  **Фишки:** Котики для опаздунов 🙀.
5.  **Контроль:** Тест на группах "404" (теряются) и "408" (прогуливают).
6.  **Техно-требования:** Python 2.4 (!!!), Документация, Архитектура.
---

## 2. 📞 Звонок с Заказчиком

Вы прочитали письмо. У вас куча вопросов (я надеюсь).
Сейчас я — **Представитель Ректората**. Я не разбираюсь в коде, но я плачу деньги.

**Ваша задача:**
Задать мне вопросы в чат, чтобы уточнить ТЗ.
Если вы не зададите вопрос сейчас, вам придется кодить то, что написано в письме.

## 3. Разбор полетов: Что вы упустили?

Давайте посмотрим, где вы спасли проект, а где подписали себе смертный приговор.

### 🚩 Ловушка №1: Python 2.4
Если вы согласились на это — вы уволены.
* **Почему:** Библиотеки `pandas`, `telebot`, `flask` не работают на версии 2.4.
* **Решение:** Мы берем на себя хостинг или убеждаем заказчика развернуть Docker. Мы пишем на **Python 3.10+**.

### 🚩 Ловушка №2: "Котики для опаздунов"
Заказчик хочет "результат" (котика), но не сказал "как".
* **Реализация:** Нам нужна механика **Check-in**. Бот присылает уведомление с кнопкой "Я на месте".
* **Логика:**
    1. Пара в 9:00.
    2. В 8:00 уведомление "Скоро пара".
    3. В 9:05, если кнопка не нажата -> Шлем Котика "Ты где??".
    4. Если это группа "408" (прогульщики) -> Шлем уведомление Преподавателю.

### 🚩 Ловушка №3: "Загрузка в групповые чаты"
В письме сказано: *"должен загружаться как в групповые, так и в личные чаты"*.
* **Риск:** Вася из группы 404 по приколу загрузит расписание, где пар нет. Бот поверит и отменит пары всему ВУЗу.
* **Решение:** Разграничение прав. Загружать файл может только **Админ** (через веб-интерфейс или спец-пароль).

---
**Итоговое решение (Architecture Decision):**
Мы делим систему на две части:
1.  **Backend (Flask):** Веб-сайт для методиста (Загрузка CSV, управление).
2.  **Frontend (Telegram Bot):** Для студентов (Просмотр, напоминания, котики).

 Ловушка "CSV" и Data Engineering

##  Вскрываем "черный ящик" заказчика

Помните, в письме было сказано: *"Возможность загрузки файла в формате CSV"*?
Мы запросили пример файла. И вот что нам прислали:
`МШИБС_расписание_2025...xlsx`

Взгляните на структуру:
1.  **Объединенные ячейки (Merged Cells):** Даты и Дни недели объединены на несколько строк.
2.  **Шапка где-то в середине:** Заголовки групп начинаются только на 7-й строке.
3.  **Визуальный мусор:** Пустые строки, надписи "Утверждаю".

###  Почему CSV нас убил бы?
CSV (Comma Separated Values) — это простой текст. При сохранении этого Экселя в CSV:
* Объединенные ячейки "рассыпаются". Данные остаются только в левой верхней ячейке, остальные становятся пустыми.
* Мы потеряли бы связь между "Временем" и "Предметом", потому что время написано один раз на 4 строки.

**Вывод:** Мы работаем только с `.xlsx` и библиотекой `openpyxl`.


# Этап 2. Бюрократия, которая спасает жизнь (Documentation)

🛑 **СТОП! УБЕРИТЕ РУКИ ОТ КЛАВИАТУРЫ!**

Мы поговорили с заказчиком. Мы поняли его боль. Мы увидели страшный Excel-файл.
Если вы сейчас начнете писать код, через неделю вы будете его переписывать, потому что:
1.  Забудете про кнопку "Отмена".
2.  Не продумаете, где хранить список пользователей.
3.  Напишете парсер, который падает от объединенных ячеек.

Сейчас мы составим **4 главных документа** (Артефакта) нашего проекта. Без них разработк

## 📄 Артефакт №1: Функциональные требования (FR)

Мы переводим "человеческий язык" заказчика в список функций программы.

**Роли пользователей:**
1.  **Студент** (Потребитель контента).
2.  **Администратор** (Учебная часть/Методист).

**Таблица функций (MVP - Minimum Viable Product):**

| ID | Роль | Функция (User Story) | Критерий приемки (Acceptance Criteria) |
|:---|:---|:---|:---|
| **F-01** | Студент | Регистрация (выбор группы) | Студент нажимает `/start`, бот предлагает список групп кнопками. Выбор сохраняется. |
| **F-02** | Студент | Получение расписания на сегодня | По команде `/today` бот выдает список пар на текущую дату для группы студента. |
| **F-03** | Студент | Получение расписания на завтра | По команде `/tomorrow` бот выдает пары на следующий день. |
| **F-04** | Админ | Загрузка файла расписания | Через веб-интерфейс можно загрузить `.xlsx` файл. Старый файл заменяется новым. |
| **F-05** | Система | Автоматическое обновление | После загрузки файла бот сразу начинает использовать новые данные без перезагрузки скрипта. |



## 📄 Артефакт №1: Функциональные требования (FR) — Версия 2.0 (Исправленная)

Мы пересмотрели требования и поняли, что забыли ключевых стейкхолдеров — **Преподавателей**.
Плюс, текстовые команды — это неудобно. Мы требуем **Кнопки**.

**Роли пользователей:**
1.  **Студент** (Получает знания и пинки).
2.  **Преподаватель** (Получает списки прогульщиков и расписание).
3.  **Админ** (Управляет файлом).

### Таблица требований (MVP)

| ID | Роль | Функция (User Story) | Детали и Критерии приемки |
|:---|:---|:---|:---|
| **FR-01** | Студент | **Регистрация** | При команде `/start` бот предлагает выбрать роль. Если "Студент" — выдает клавиатуру со списком групп из Excel. |
| **FR-02** | Преподаватель | **Регистрация** | Если роль "Преподаватель" — бот предлагает выбрать ФИО из списка (список берется из того же Excel). |
| **FR-03** | Все | **Просмотр расписания (День)** | Кнопка "На сегодня" / "На завтра". Выдает список пар текстом. |
| **FR-04** | Все | **Просмотр всего расписания** | Кнопка "Вся неделя". Выдает полное расписание для своей группы/ФИО. |
| **FR-05** | Система | **Фоновые уведомления** | Бот сам (без команды) отправляет сообщение за 1 час до начала первой пары. |
| **FR-06** | Студент | **Check-In (Отметка)** | В уведомлении о паре должна быть **Инлайн-кнопка** "Я приду". Нажатие фиксируется в системе. |
| **FR-07** | Система | **Контроль присутствия** | За 10 мин до пары бот проверяет, кто НЕ нажал кнопку, и формирует список. |
| **FR-08** | Преподаватель | **Получение отчета** | Если есть прогульщики, Преподаватель получает сообщение: *"Внимание! Эти студенты не подтвердили присутствие: [Список]"*. |
| **FR-09** | Админ | **Загрузка данных** | Веб-интерфейс для загрузки `.xlsx`. При загрузке данные в боте обновляются моментально. |

---
**Важное требование к Интерфейсу (UI/UX):**
* Минимум ввода текста с клавиатуры.
* Группы, Дни недели, Действия — всё через **Reply Keyboard** (Кнопки внизу экрана).
* Подтверждение присутствия — через **Inline Keyboard** (Кнопка под сообщением), чтобы не засорять чат.

##  Почему мы добавили эти пункты?

1.  **FR-02 (Преподаватели):** Без регистрации преподавателя бот не будет знать `chat_id`, куда отправлять жалобу на прогульщиков. Бот не может писать первым по юзернейму, он может только *отвечать* зарегистрированному пользователю.
2.  **FR-06 (Кнопка в уведомлении):** Это то самое требование "Котики/Контроль". Мы не можем просто верить студенту, мы требуем действия (Action).
3.  **UI Buttons:** Студенты ленивы. Набирать `БИ-21-2` руками — это риск опечатки. Нажать кнопку — надежно.

##  Артефакт №2: Архитектура Системы (High-Level Design)

Мы строим систему по принципу **"Монолит с общим файловым хранилищем"**.

Это значит, что у нас будет **Один Сервер** (ваш ноутбук или облачный VPS), на котором крутятся три независимых процесса, использующих одну и ту же папку с данными.

###  Схема взаимодействия компонентов



Давайте разберем эту схему по кирпичикам:

1.  **Frontend №1: Telegram App (Для Пользователей)**
    * Это интерфейс Студента и Преподавателя.
    * Они нажимают кнопки, данные летят на сервер Telegram, а оттуда — к нам в Python-скрипт (через Long Polling).

2.  **Frontend №2: Web Admin Panel (Для Заказчика)**
    * Это веб-сайт (Flask), доступный через браузер.
    * Иван Иванович заходит по адресу `http://our-server:5000/upload`.
    * Единственная задача этого компонента — безопасно принять файл и положить его в папку `data/`.

3.  **Data Layer (Слой Данных)**
    * У нас нет базы данных MySQL или PostgreSQL. Для MVP это избыточно.
    * Нашей "Базой Данных" служат два файла на жестком диске:
        * `schedule.xlsx` — Источник истины о расписании. Обновляется Админом.
        * `users.json` — Реестр пользователей. Обновляется Ботом (при регистрации).

4.  **Backend Logic (Мозг)**
    * **Parser Service:** Превращает Excel в словарь Python.
    * **Bot Handler:** Отвечает на команды `/start`, нажатия кнопок.
    * **Scheduler (Будильник):** Фоновый процесс, который раз в минуту просыпается и проверяет: "Не пора ли кого-то уведомить?".

### 🔄 Потоки Данных (Data Flow)

Чтобы понять архитектуру, проследим путь данных.

#### Сценарий А: "Обновление расписания"
1.  **Админ** загружает файл `new_schedule.xlsx` через Браузер.
2.  **Flask** сохраняет его в папку `data/schedule.xlsx` (перезаписывая старый).
3.  **Flask** дает сигнал (или Бот сам при следующем запросе) перечитать файл.
4.  **Бот** видит новые данные.

#### Сценарий Б: "Уведомление и Стук"
1.  **Scheduler** (внутри Бота) видит: "Время 08:30".
2.  Читает `schedule.xlsx` -> "Ага, у группы БИ-21 пара в 09:00".
3.  Читает `users.json` -> "В группе БИ-21 есть id123, id456".
4.  **Бот** отправляет им сообщение с кнопкой [Я приду].
5.  **Студент** жмет кнопку. Бот помечает в памяти: `check_in[id123] = True`.
6.  В 08:50 **Scheduler** проверяет: "id456 не нажал".
7.  Ищет в Excel фамилию препода -> Ищет в `users.json` chat_id препода -> Шлет жалобу.

### 🛠 Технологический Стек (Tech Stack)

Мы фиксируем инструменты, чтобы в середине разработки кто-то не притащил лишнюю библиотеку.

| Компонент | Технология | Почему выбрали? |
|:---|:---|:---|
| **Язык** | Python 3.10+ | Потому что заказчик хотел 2.4, но мы его переубедили. |
| **Веб-сервер** | **Flask** | Легкий, требует мало кода, идеально для простой админки. |
| **Бот** | **pyTelegramBotAPI (Telebot)** | Простая синхронная библиотека, отлично подходит для новичков (в отличие от aiogram). |
| **Работа с Excel** | **openpyxl + Pandas** | Pandas нужен для умной фильтрации, openpyxl — для чтения объединенных ячеек. |
| **Расписание** | **schedule / threading** | Для запуска фоновых задач (напоминаний). |
| **Хранение** | **JSON + FileSystem** | Самый быстрый способ реализации (Zero-configuration). |

---
**Риски архитектуры:**
* *Проблема одновременного доступа:* Что будет, если Админ загружает файл ровно в ту секунду, когда Бот его читает? (Файл может быть битым).
* *Решение:* Мы будем использовать простой `try-except` при чтении файла. Если файл занят — попробуем через секунду.

##  Артефакт №3: Контракт Данных (Data Schema)

Мы договорились про Excel. Но нам нужен еще один файл, о котором заказчик даже не думал.
Нам нужно где-то хранить список: **Кто есть Кто**.

Мы не используем SQL (слишком сложно для старта), мы используем **JSON**.
Это будет файл `users.json`, который бот создаст сам.

### Структура `users.json`

```json
{
  "123456789": {
    "role": "student",
    "group": "МИСТ-25-1",
    "notifications": true,
    "last_active": "2023-10-01"
  },
  "987654321": {
    "role": "teacher",
    "name": "Иванов И.И.",
    "notifications": true
  }
}

Правила работы с этим файлом:

Ключ: Это всегда chat_id (уникальный номер пользователя в Telegram).

Role: Обязательное поле. Либо student, либо teacher.

Group / Name: Зависит от роли. Студент привязан к группе, Преподаватель — к Фамилии в расписании.

Notifications: Флаг (True/False). Если False — мы не шлем этому человеку спам утром.

Почему это важно? Без этого файла, каждый раз при перезагрузке бота, все студенты "забывались" бы, и им приходилось бы регистрироваться заново.

##  Артефакт №4: ТКП (Технико-Коммерческое Предложение)

Это финальный документ. Мы отправляем его Ивану Ивановичу.
Если он пишет "ОК" — мы начинаем работать. Если нет — мы идем переделывать схему.

---
### 📄 КОММЕРЧЕСКОЕ ПРЕДЛОЖЕНИЕ
**Исполнитель:** Horns & Hooves IT
**Заказчик:** Ректорат ВУЗа

**1. Предлагаемое решение:**
Автоматизированная система информирования "Староста-Бот 3000".
Система состоит из Telegram-бота для студентов и Веб-панели для методистов.

**2. Стек технологий (Внимание!):**
* Язык программирования: **Python 3.10+** (Мы отказываемся от Python 2.4 ввиду невозможности реализации современных требований безопасности).
* Хостинг: Предоставляется заказчиком или облачное решение.

**3. Этапы реализации и Сроки:**
* **Этап 1 (Ядро):** Парсинг Excel, Админка, Регистрация студентов. (Срок: 3 дня)
* **Этап 2 (Бот):** Вывод расписания, кнопки меню. (Срок: 2 дня)
* **Этап 3 (Логика):** Уведомления, "Стукач" (уведомления преподам). (Срок: 5 дней)
* **Итого:** 2 недели до запуска MVP.

**4. Стоимость (в часах):**
* Аналитика и проектирование: 10 часов.
* Разработка Backend (Flask + Parser): 15 часов.
* Разработка Bot Logic: 20 часов.
* Тестирование (на группе 404): 5 часов.
* **Итого:** 50 нормо-часов.

**5. Ограничения (Out of Scope):**
* Мы НЕ делаем интеграцию с 1С ВУЗа.
* Мы НЕ несем ответственности, если студент передал телефон другу, чтобы тот нажал кнопку "Я пришел".

##  Момент Истины

Мы отправляем письмо.
*(Представьте звук отправки email)*.

...
...

**Входящее сообщение от Иван Ивановича:**
> *"Коллеги, добрый день!
> Цену увидел, поперхнулся чаем, но согласовал.
> Про Питон 2.4 — сисадмин Петрович расстроился, но сказал, что выделит вам виртуалку с новым Убунту.
> Начинайте. Жду прототип в понедельник.
> И не забудьте про котиков!"*

---
**КОНТРАКТ ПОДПИСАН!**
Только теперь, имея на руках ТЗ, Схему Данных и ТКП, мы открываем IDE.

#  Часть 3: Пишем код. Слой Данных (The Engine)

Мы начинаем не с кнопок, а с "мозгов".
Наша задача: Превратить файл `МШИБС_расписание...xlsx` в удобный словарь Python.

Мы используем библиотеку `openpyxl`, потому что она умеет работать с форматированием (нам важно знать, где объединены ячейки).

### 🛠 Создаем файл `schedule_parser.py`

В Jupyter Notebook мы можем создать файл командой `%%writefile`.
Этот класс будет делать всю грязную работу: искать заголовки, разбирать объединенные даты и игнорировать "Обеды".

##  Археология кода (Legacy Code)

Писать парсер для такой сложной таблицы с нуля — это 2 часа боли.
Но вам повезло. Я нашел на старом сервере кусок кода, который писали студенты в прошлом году для похожей задачи.

Давайте выделим из него **чистую логику парсинга**

In [7]:
%%writefile schedule_parser.py
from openpyxl import load_workbook
from openpyxl.utils.cell import range_boundaries
from datetime import datetime

class ScheduleParser:
    def __init__(self, file_path):
        self.file_path = file_path
        self.schedule_data = {} # Сюда сложим результат

    def parse_date_time(self, date_obj, time_str):
        """Превращает дату и строку '10:00-11:25' в datetime объекты"""
        try:
            if isinstance(date_obj, datetime):
                date_str = date_obj.date().strftime('%Y-%m-%d')
            else:
                # Иногда дата может прилететь строкой, добавим страховку
                date_str = str(date_obj).split()[0]

            start_time, end_time = time_str.split('-')
            start_dt = datetime.strptime(f"{date_str} {start_time.strip()}", "%Y-%m-%d %H:%M")
            end_dt = datetime.strptime(f"{date_str} {end_time.strip()}", "%Y-%m-%d %H:%M")
            return start_dt, end_dt
        except Exception as e:
            print(f"⚠️ Ошибка времени: {e}")
            return None, None

    def parse(self):
        print(f"📂 Открываю файл: {self.file_path}")
        try:
            workbook = load_workbook(self.file_path, data_only=True) # data_only=True важен для формул!
            sheet = workbook.active
        except FileNotFoundError:
            print("❌ Файл не найден!")
            return {}

        # 1. Ищем заголовки групп (строка 7 по ТЗ)
        # sheet.iter_rows возвращает генератор, берем нужную строку
        headers = []
        for cell in sheet[7]: 
            headers.append(cell.value)

        # Определяем, в каких колонках живут группы (исключаем "День", "Время" и т.д.)
        group_columns = {} # {index: 'Group_Name'}
        ignore_list = ["неделя", "день недели", "дата", "пара", "время"]
        
        for idx, header in enumerate(headers):
            if header and str(header).lower() not in ignore_list:
                group_columns[idx] = header
        
        print(f"✅ Нашел группы: {list(group_columns.values())}")

        # Хелпер для объединенных ячеек (Магия OpenPyXL)
        def get_value(r, c):
            cell = sheet.cell(r, c + 1) # openpyxl индексы с 1
            for merged in sheet.merged_cells.ranges:
                if cell.coordinate in merged:
                    # Если ячейка часть объединения - берем значение из левой верхней
                    return sheet.cell(merged.min_row, merged.min_col).value
            return cell.value

        # 2. Бежим по строкам (начинаем с 8-й)
        for row_idx in range(8, sheet.max_row + 1):
            # Колонки фиксированы по ТЗ: 2-День, 3-Дата, 5-Время
            day = get_value(row_idx, 1)      # Col B (index 1)
            raw_date = get_value(row_idx, 2) # Col C (index 2)
            raw_time = sheet.cell(row_idx, 5).value # Col E

            # Пропуски и проверки
            if not raw_date or not raw_time: continue
            if "обед" in str(raw_time).lower(): continue

            start, end = self.parse_date_time(raw_date, raw_time)
            if not start: continue

            # 3. Собираем пары для каждой найденной группы
            for col_idx, group_name in group_columns.items():
                subject = get_value(row_idx, col_idx)
                
                if subject and str(subject).strip():
                    if group_name not in self.schedule_data:
                        self.schedule_data[group_name] = []
                    
                    self.schedule_data[group_name].append({
                        'day': day,
                        'start': start,
                        'end': end,
                        'subject': subject.strip()
                    })
        
        print(f"🎉 Парсинг завершен. Обработано групп: {len(self.schedule_data)}")
        return self.schedule_data

In [18]:
%%writefile schedule_parser.py
from openpyxl import load_workbook
from openpyxl.utils.cell import range_boundaries
from datetime import datetime

class ScheduleParser:
    def __init__(self, file_path):
        self.file_path = file_path
        self.schedule_data = {} # Итоговый словарь

    def parse(self):
        """Главный метод: читает файл и возвращает словарь расписания"""
        print(f"📂 Загружаю файл: {self.file_path}")
        try:
            # data_only=True позволяет читать значения формул, а не сами формулы
            workbook = load_workbook(self.file_path, data_only=True)
            sheet = workbook.active
        except FileNotFoundError:
            print("❌ Файл не найден!")
            return {}

        # 1. Анализируем заголовки (Строка 7 по ТЗ)
        # Нам нужно понять, в какой колонке какая группа живет
        headers = [cell.value for cell in sheet[7]] # Читаем 7-ю строку
        group_columns = {}
        
        # Список слов-исключений (это не названия групп)
        ignore_list = ["неделя", "день недели", "дата", "пара", "время", None]
        
        for idx, header in enumerate(headers):
            # Если в заголовке есть текст и это не служебное слово — значит это Группа
            if header and str(header).strip().lower() not in ignore_list:
                group_columns[idx] = header.strip()
        
        print(f"✅ Найдены группы: {list(group_columns.values())}")

        # Хелпер для чтения объединенных ячеек
        # Если ячейка часть merge-блока, значение лежит только в левой верхней клетке
        def get_value(r, c):
            cell = sheet.cell(r, c + 1) # openpyxl использует индексы с 1
            for merged in sheet.merged_cells.ranges:
                if cell.coordinate in merged:
                    return sheet.cell(merged.min_row, merged.min_col).value
            return cell.value

        # 2. Итерируемся по строкам с данными (начиная с 8-й)
        for row_idx in range(8, sheet.max_row + 1):
            # Жесткая привязка к колонкам по ТЗ:
            # Col 2 (idx 1) = День недели, Col 3 (idx 2) = Дата, Col 5 (idx 4) = Время
            raw_date = get_value(row_idx, 2) 
            raw_time = sheet.cell(row=row_idx, column=5).value

            # Пропускаем пустые строки и Обеды
            if not raw_date or not raw_time: continue
            if "обед" in str(raw_time).lower(): continue

            # Парсим время и дату
            start_dt, end_dt = self._parse_datetime(raw_date, raw_time)
            if not start_dt: continue

            # 3. Собираем пары для каждой найденной группы
            for col_idx, group_name in group_columns.items():
                subject = get_value(row_idx, col_idx)
                
                # Если в ячейке есть текст — значит есть пара
                if subject and str(subject).strip():
                    if group_name not in self.schedule_data:
                        self.schedule_data[group_name] = []
                    
                    self.schedule_data[group_name].append({
                        'start': start_dt,
                        'end': end_dt,
                        'subject': subject.strip()
                    })
        
        print(f"🎉 Обработано {len(self.schedule_data)} групп")
        return self.schedule_data

    def _parse_datetime(self, date_obj, time_str):
        """Превращает дату и строку времени (10:00-11:25) в datetime"""
        try:
            # Если дата пришла как datetime (из экселя), берем дату строкой
            if isinstance(date_obj, datetime):
                date_str = date_obj.strftime('%Y-%m-%d')
            else:
                date_str = str(date_obj).split()[0] # Отрезаем время если есть

            start_t, end_t = time_str.split('-')
            start_dt = datetime.strptime(f"{date_str} {start_t.strip()}", "%Y-%m-%d %H:%M")
            end_dt = datetime.strptime(f"{date_str} {end_t.strip()}", "%Y-%m-%d %H:%M")
            return start_dt, end_dt
        except Exception:
            return None, None

Writing schedule_parser.py


In [6]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


## 3. Тестируем "Мозги" (Data check)

Прежде чем прикручивать Телеграм, мы должны убедиться, что парсер работает корректно.
**Никогда** не пишите бота, пока не проверили логику на чистом Python.


In [19]:
# Импортируем наш свежий класс
from schedule_parser import ScheduleParser
import pprint # Для красивого вывода словарей

# Укажите точное имя файла, который лежит рядом с тетрадкой!
FILE_NAME = "МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx" 

parser = ScheduleParser(FILE_NAME)
data = parser.parse()

# Выведем расписание для первой попавшейся группы
if data:
    first_group = list(data.keys())[0]
    print(f"\n🔎 Расписание для {first_group} (первые 3 пары):")
    pprint.pprint(data[first_group][:10])
else:
    print("Данные пустые. Проверьте имя файла или структуру!")

📂 Загружаю файл: МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx
✅ Найдены группы: ['МИСТ-25-1-1', 'МИСТ-25-1-2', 'МИСТ-25-2-1', 'МИСТ-25-2-2']
🎉 Обработано 4 групп

🔎 Расписание для МИСТ-25-1-1 (первые 3 пары):
[{'end': datetime.datetime(2025, 12, 1, 11, 25),
  'start': datetime.datetime(2025, 12, 1, 10, 0),
  'subject': 'Дистанционно. Архитектуры систем хранения данных. Дюмин '
             'Александр Александрович (Пр5-6)'},
 {'end': datetime.datetime(2025, 12, 1, 13, 0),
  'start': datetime.datetime(2025, 12, 1, 11, 35),
  'subject': 'Дистанционно. Архитектуры систем хранения данных. Дюмин '
             'Александр Александрович (Пр5-6)'},
 {'end': datetime.datetime(2025, 12, 2, 20, 0),
  'start': datetime.datetime(2025, 12, 2, 18, 30),
  'subject': 'Дистанционно. Новые направления и технологии современных СУБД '
             '(Пр5-6). Ривкин Марк Нахимович'},
 {'end': datetime.datetime(2025, 12, 2, 21, 30),
  'start': datetime.datetime(2025, 12, 2, 20, 10),
  'subject': 'Дистанционн

# Этап 3.1: Data Cleaning. Почему наш парсер плохой?

Мы запустили код и получили данные. Казалось бы, победа?
Давайте посмотрим внимательнее на то, что лежит в поле `subject`:

> `'Дистанционно. Архитектуры систем хранения данных. Дюмин Александр Александрович (Пр5-6)'`

> `'Проектирование информационных систем. (Пр1-2). Акатова Наталья Анатольевна МШ ИБС ауд.103'`

**Проблема:**
Мы свалили всё в кучу: формат ("Дистанционно"), предмет, преподавателя и аудиторию ("ауд. 103").

**Почему это блокирует задачу?**
В ТЗ сказано: *"Отправлять уведомление преподавателю"*.
А как мы найдем преподавателя? У нас нет поля `teacher`. Если мы попробуем искать пользователя с именем "Дистанционно. Архитектуры...", мы никого не найдем.

**Задача:** Нам нужно "разрезать" эту строку на три части:
1.  **Subject:** "Архитектуры систем хранения данных"
2.  **Teacher:** "Дюмин Александр Александрович"
3.  **Room:** "МШ ИБС ауд. 103" (если есть)

Для этого нам понадобятся **Регулярные выражения (Regular Expressions)**.

##  Введение в Regex (re)

Регулярные выражения — это язык поиска шаблонов в тексте.

Посмотрите на ФИО преподавателей: `Дюмин Александр Александрович`, `Ривкин Марк Нахимович`.
Какой у них шаблон?
1.  Слово с Большой буквы.
2.  Пробел.
3.  Слово с Большой буквы.
4.  Пробел.
5.  Слово с Большой буквы.

На языке Regex это выглядит так: `([А-ЯЁ][а-яё]+\s[А-ЯЁ][а-яё]+\s[А-ЯЁ][а-яё]+)`

А шаблон аудитории? Обычно это слово "ауд" или "МШ ИБС".
Давайте допишем в наш парсер функцию `clean_text`, которая будет вытаскивать эти данные.

In [1]:
%%writefile schedule_parser.py
from openpyxl import load_workbook
from datetime import datetime
import re # Подключаем библиотеку регулярных выражений

class ScheduleParser:
    def __init__(self, file_path):
        self.file_path = file_path
        self.schedule_data = {}

    def parse(self):
        print(f"📂 Загружаю файл: {self.file_path}")
        try:
            workbook = load_workbook(self.file_path, data_only=True)
            sheet = workbook.active
        except FileNotFoundError:
            print("❌ Файл не найден!")
            return {}

        # 1. Ищем заголовки групп (Строка 7)
        headers = [cell.value for cell in sheet[7]]
        group_columns = {}
        ignore_list = ["неделя", "день недели", "дата", "пара", "время", None]
        
        for idx, header in enumerate(headers):
            if header and str(header).strip().lower() not in ignore_list:
                group_columns[idx] = header.strip()
        
        print(f"✅ Найдены группы: {list(group_columns.values())}")

        # Хелпер для объединенных ячеек
        def get_value(r, c):
            cell = sheet.cell(r, c + 1)
            for merged in sheet.merged_cells.ranges:
                if cell.coordinate in merged:
                    return sheet.cell(merged.min_row, merged.min_col).value
            return cell.value

        # 2. Парсинг строк
        for row_idx in range(8, sheet.max_row + 1):
            raw_date = get_value(row_idx, 2) 
            raw_time = sheet.cell(row=row_idx, column=5).value

            if not raw_date or not raw_time: continue
            if "обед" in str(raw_time).lower(): continue

            start_dt, end_dt = self._parse_datetime(raw_date, raw_time)
            if not start_dt: continue

            # 3. Обработка каждой группы
            for col_idx, group_name in group_columns.items():
                raw_text = get_value(row_idx, col_idx)
                
                if raw_text and str(raw_text).strip():
                    # ТУТ МАГИЯ: Разбираем грязную строку на части
                    clean_data = self._extract_details(str(raw_text).strip())
                    
                    if group_name not in self.schedule_data:
                        self.schedule_data[group_name] = []
                    
                    self.schedule_data[group_name].append({
                        'start': start_dt,
                        'end': end_dt,
                        'subject': clean_data['subject'],
                        'teacher': clean_data['teacher'], # Теперь это отдельное поле!
                        'room': clean_data['room'],       # И это тоже!
                        'raw': raw_text # Оставляем оригинал на всякий случай
                    })
        
        return self.schedule_data

    def _parse_datetime(self, date_obj, time_str):
        try:
            if isinstance(date_obj, datetime):
                date_str = date_obj.strftime('%Y-%m-%d')
            else:
                date_str = str(date_obj).split()[0]
            
            # Очистка времени от лишних пробелов (бывает "10:00 - 11:25")
            start_t, end_t = time_str.replace(' ', '').split('-')
            start_dt = datetime.strptime(f"{date_str} {start_t}", "%Y-%m-%d %H:%M")
            end_dt = datetime.strptime(f"{date_str} {end_t}", "%Y-%m-%d %H:%M")
            return start_dt, end_dt
        except Exception:
            return None, None

    def _extract_details(self, text):
        """
        Функция 'пылесос'. Пытается найти ФИО и Аудиторию в строке.
        Пример входа: "Дистанционно. Архитектуры СХД. Дюмин Александр Александрович (Пр5-6)"
        """
        result = {
            'subject': text, # По умолчанию считаем всё предметом
            'teacher': None,
            'room': None
        }

        # 1. Ищем ФИО (Фамилия Имя Отчество - 3 слова с большой буквы)
        # Паттерн: Заглавная, буквы, пробел, Заглавная, буквы, пробел, Заглавная, буквы
        # Исключаем слова типа "Дистанционно" и "Лекция", проверяя, что слова идут подряд
        teacher_pattern = r'([А-ЯЁ][а-яё]+\s[А-ЯЁ][а-яё]+\s[А-ЯЁ][а-яё]+)'
        match = re.search(teacher_pattern, text)
        
        if match:
            result['teacher'] = match.group(1)
            # Удаляем ФИО из названия предмета, чтобы не дублировать
            result['subject'] = text.replace(result['teacher'], '').strip()

        # 2. Ищем аудиторию (Паттерн: "ауд." или "МШ ИБС" + цифры)
        # Это упрощенный поиск, можно усложнять бесконечно
        if "ауд." in text or "МШ ИБС" in text:
            # Если есть явное указание локации, пробуем вырезать конец строки или искать ключевые слова
            # Для MVP просто оставим это в subject, но если найдем "МШ ИБС" - вынесем
            room_match = re.search(r'(МШ ИБС.*|ауд\..*)', text)
            if room_match:
                result['room'] = room_match.group(1)
                # Убираем аудиторию из предмета
                result['subject'] = result['subject'].replace(result['room'], '').strip()
        else:
            result['room'] = "Не указана (возможно, Дистанционно)"

        # 3. Чистим мусор (лишние точки и скобки в начале/конце)
        result['subject'] = result['subject'].strip(' .,')
        
        return result

Overwriting schedule_parser.py


In [2]:
from schedule_parser import ScheduleParser
import pprint

# Файл тот же
FILE_NAME = "МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx"

parser = ScheduleParser(FILE_NAME)
data = parser.parse()

if data:
    first_group = list(data.keys())[0]
    print(f"\n🔎 ПРОВЕРКА ДАННЫХ ДЛЯ: {first_group}")
    
    # Выводим первые 5 пар, чтобы убедиться, что Teacher отделился
    for lesson in data[first_group][:5]:
        print(f"⏰ {lesson['start'].strftime('%d.%m %H:%M')}")
        print(f"   📚 Предмет: {lesson['subject']}")
        print(f"   👨‍🏫 Препод: {lesson['teacher']}") 
        print(f"   🚪 Аудит.:  {lesson['room']}")
        print("-" * 30)
else:
    print("Данные не найдены.")

📂 [Parser] Читаю файл: МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx
✅ [Parser] Готово. Групп: 4

🔎 ПРОВЕРКА ДАННЫХ ДЛЯ: МИСТ-25-1-1
⏰ 01.12 10:00
   📚 Предмет: Архитектуры систем хранения данных. (Пр5-6
   👨‍🏫 Препод: Дюмин Александр Александрович
   🚪 Аудит.:  Дистанционно
------------------------------
⏰ 01.12 11:35
   📚 Предмет: Архитектуры систем хранения данных. (Пр5-6
   👨‍🏫 Препод: Дюмин Александр Александрович
   🚪 Аудит.:  Дистанционно
------------------------------
⏰ 02.12 01:30
   📚 Предмет: Новые направления и технологии современных СУБД (Пр5-6
   👨‍🏫 Препод: Ривкин Марк Нахимович
   🚪 Аудит.:  Дистанционно
------------------------------
⏰ 02.12 20:10
   📚 Предмет: Новые направления и технологии современных СУБД (Пр5-6
   👨‍🏫 Препод: Ривкин Марк Нахимович
   🚪 Аудит.:  Дистанционно
------------------------------
⏰ 03.12 10:00
   📚 Предмет: Практика моделирования бизнес-процессов (Пр12-13
   👨‍🏫 Препод: Вагнер Юлия Борисовна
   🚪 Аудит.:  Дистанционно
----------------------

#  Этап 3.2: Bugfix. Работа над ошибками

Мы запустили код и получили `KeyError`. Это нормально.
Более того, мы увидели, что данные сложнее, чем мы думали.

**Анализ новых данных (кейсы, которые сломали парсер):**
1.  `"Дистанционно. Архитектуры... Дюмин Александр Александрович..."`
    * *Вывод:* Слово "Дистанционно" стоит в начале. Это по сути **Аудитория**.
2.  `"...Пономарева Юлия Павловна (Пр14) МШ ИБС ауд. 103"`
    * *Вывод:* Аудитория может быть в конце.
3.  `"Дистанционно... Ривкин Марк Нахимович"`
    * *Вывод:* Преподаватель может быть в самом конце строки.

**Задача:** Переписать функцию `_extract_details` так, чтобы она:
1.  Всегда возвращала ключи `teacher` и `room` (чтобы не было `KeyError`).
2.  Понимала, что "Дистанционно" — это тоже локация.
3.  Надежно вырезала ФИО.



Переписываем `schedule_parser.py` с учетом этих правил.

In [3]:
%%writefile schedule_parser.py
from openpyxl import load_workbook
from datetime import datetime
import re

class ScheduleParser:
    def __init__(self, file_path):
        self.file_path = file_path
        self.schedule_data = {}

    def parse(self):
        print(f"📂 [Parser] Читаю файл: {self.file_path}")
        try:
            workbook = load_workbook(self.file_path, data_only=True)
            sheet = workbook.active
        except FileNotFoundError:
            print("❌ Файл не найден!")
            return {}

        # 1. Заголовки (Строка 7)
        headers = [cell.value for cell in sheet[7]]
        group_columns = {}
        ignore_list = ["неделя", "день недели", "дата", "пара", "время", None]
        
        for idx, header in enumerate(headers):
            if header and str(header).strip().lower() not in ignore_list:
                group_columns[idx] = header.strip()
        
        # 2. Парсинг строк
        for row_idx in range(8, sheet.max_row + 1):
            # Хелпер для merged cells
            def get_val(r, c):
                cell = sheet.cell(r, c + 1)
                for merged in sheet.merged_cells.ranges:
                    if cell.coordinate in merged:
                        return sheet.cell(merged.min_row, merged.min_col).value
                return cell.value

            raw_date = get_val(row_idx, 2) 
            raw_time = sheet.cell(row=row_idx, column=5).value

            if not raw_date or not raw_time: continue
            if "обед" in str(raw_time).lower(): continue

            start_dt, end_dt = self._parse_time(raw_date, raw_time)
            if not start_dt: continue

            # 3. Обработка групп
            for col_idx, group_name in group_columns.items():
                raw_text = get_val(row_idx, col_idx)
                
                if raw_text and str(raw_text).strip():
                    details = self._extract_details(str(raw_text).strip())
                    
                    if group_name not in self.schedule_data:
                        self.schedule_data[group_name] = []
                    
                    self.schedule_data[group_name].append({
                        'start': start_dt,
                        'end': end_dt,
                        'subject': details['subject'],
                        'teacher': details['teacher'],
                        'room': details['room'],
                        'raw': raw_text
                    })
        
        print(f"✅ [Parser] Готово. Групп: {len(self.schedule_data)}")
        return self.schedule_data

    def _parse_time(self, date_obj, time_str):
        try:
            if isinstance(date_obj, datetime):
                d_str = date_obj.strftime('%Y-%m-%d')
            else:
                d_str = str(date_obj).split()[0]
            
            s_t, e_t = time_str.replace(' ', '').split('-')
            start = datetime.strptime(f"{d_str} {s_t}", "%Y-%m-%d %H:%M")
            end = datetime.strptime(f"{d_str} {e_t}", "%Y-%m-%d %H:%M")
            return start, end
        except:
            return None, None

    def _extract_details(self, text):
        res = {'subject': text, 'teacher': 'Не указан', 'room': 'Не указана'}
        
        # 1. ФИО: 3 слова с Большой буквы
        teacher_match = re.search(r'([А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+\s+[А-ЯЁ][а-яё]+)', text)
        if teacher_match:
            res['teacher'] = teacher_match.group(1)
            res['subject'] = res['subject'].replace(res['teacher'], '')

        # 2. Аудитория
        if "Дистанционно" in text:
            res['room'] = "Дистанционно"
            res['subject'] = res['subject'].replace("Дистанционно", "")
        
        room_match = re.search(r'(МШ ИБС.*|ауд\.\s*\d+)', text)
        if room_match:
            val = room_match.group(1)
            res['room'] = f"{res['room']}, {val}" if res['room'] != 'Не указана' else val
            res['subject'] = res['subject'].replace(val, "")

        # 3. Чистка
        res['subject'] = re.sub(r'\s+', ' ', res['subject']).strip(" .,()")
        return res

Overwriting schedule_parser.py


In [4]:
import importlib
import schedule_parser
importlib.reload(schedule_parser)

from schedule_parser import ScheduleParser
import pprint

FILE_NAME = "МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx"

parser = ScheduleParser(FILE_NAME)
data = parser.parse()

if data:
    first_group = list(data.keys())[3]
    print(f"\n🔎 ПРОВЕРКА (v2) ДЛЯ: {first_group}")
    
    # Берем первые 5 пар
    for lesson in data[first_group][:10]:
        print(f"⏰ {lesson['start'].strftime('%d.%m %H:%M')}")
        print(f"   📚 Предмет: {lesson['subject']}")
        print(f"   👨‍🏫 Препод: {lesson['teacher']}") 
        print(f"   🚪 Аудит.:  {lesson['room']}")
        print("-" * 30)
else:
    print("Данные не найдены.")

📂 [Parser] Читаю файл: МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx
✅ [Parser] Готово. Групп: 4

🔎 ПРОВЕРКА (v2) ДЛЯ: МИСТ-25-2-2
⏰ 01.12 10:00
   📚 Предмет: Архитектуры систем хранения данных. (Пр5-6
   👨‍🏫 Препод: Дюмин Александр Александрович
   🚪 Аудит.:  Дистанционно
------------------------------
⏰ 01.12 11:35
   📚 Предмет: Архитектуры систем хранения данных. (Пр5-6
   👨‍🏫 Препод: Дюмин Александр Александрович
   🚪 Аудит.:  Дистанционно
------------------------------
⏰ 01.12 19:00
   📚 Предмет: Спецглавы математики. Часть1. Контрольная работа. (Пр14
   👨‍🏫 Препод: Пономарева Юлия Павловна
   🚪 Аудит.:  МШ ИБС ауд. 103
------------------------------
⏰ 02.12 01:30
   📚 Предмет: Языки программирования для работы с большими данными (Пр7-8
   👨‍🏫 Препод: Воробьев Даниил Анатольевич
   🚪 Аудит.:  МШ ИБС ауд. 103
------------------------------
⏰ 02.12 20:10
   📚 Предмет: Языки программирования для работы с большими данными (Пр7-8
   👨‍🏫 Препод: Воробьев Даниил Анатольевич
   🚪 Аудит.: 

#  Часть 4. Computer Science для чайников: Очереди и Стеки

Прежде чем писать `bot.polling()`, давайте разберемся, как вообще работают программы, которые общаются с внешним миром.

Представьте, что ваш бот — это кассир в Макдональдсе.
Одновременно ему пишут 1000 студентов: *"Дай расписание!"*.
Если бот попытается ответить всем одновременно, у него "лопнет голова" (процессор сгорит).

Чтобы этого не случилось, в программировании придумали структуры данных.

## 1. Очередь (Queue) — FIFO

**FIFO** = First In, First Out (Первый пришел — первый ушел).

Это как живая очередь в кассу.
1.  Студент А написал "Привет" (в 10:00:01).
2.  Студент Б написал "Как дела" (в 10:00:02).

Сообщения попадают в **Очередь Телеграма**. Ваш бот забирает их по одному. Он **никогда** не обработает сообщение студента Б раньше, чем студента А.

[Image of FIFO queue diagram computer science]

> **Где это в боте?**
> Когда вы запускаете `bot.polling()`, вы запускаете бесконечный цикл, который говорит серверу Telegram: *"Дай мне первое сообщение из очереди. Обработал? Дай следующее"*.

## 2. Стек (Stack) — LIFO

**LIFO** = Last In, First Out (Последний пришел — первый ушел).

Представьте стопку тарелок. Вы кладете тарелку сверху. Чтобы добраться до нижней, нужно снять верхнюю.
Или магазин автомата (рожок): последний патрон вылетает первым.

[Image of stack data structure LIFO diagram]

> **Где это в боте?**
> Это **Навигация**.
> 1. Главное меню (Внизу стека).
> 2. Вы нажали "Расписание" -> Сверху положили экран расписания.
> 3. Вы нажали "Назад" -> Выкинули верхний экран, вернулись в меню.

В браузере кнопка "Назад" работает именно по принципу Стека.

#  Архитектура: Long Polling

Как бот узнает, что ему написали? Есть два пута:
1.  **Webhook (Стук в дверь):** Телеграм сам стучится к нам на сервер. (Сложно, нужен HTTPS, белый IP).
2.  **Long Polling (А мы не приехали?):** Бот сам постоянно спрашивает Телеграм.

Мы используем **Polling**.
Это выглядит так:
* **Бот:** "Есть сообщения?"
* **Telegram:** "Нет, жди..." (держит соединение открытым 20 секунд).
* *...прошло 5 секунд, кто-то написал...*
* **Telegram:** "Есть! Вот тебе JSON c сообщением".
* **Бот:** "Ок, обрабатываю. Есть еще?"

Это создает иллюзию мгновенного ответа, хотя на самом деле бот просто очень часто спрашивает.

#  Код Бота (Bot Logic)

Мы переходим к коду самого бота.
Вспомним наши требования из **Артефакта №3 (Архитектура)**:

1.  **Безопасность:** Токен не храним в коде (используем `.env`).
2.  **Персистентность (Память):** Бот должен помнить студентов даже после перезагрузки сервера.
    * *Решение:* Используем модуль `json` для сохранения базы пользователей на диск.
3.  **Логирование:** Используем `logging` вместо `print`, чтобы видеть время событий и уровни ошибок.

### 🛠 Подготовка: `users.json`

Нам понадобятся две вспомогательные функции:
* `load_users()`: При старте бота читает файл в переменную.
* `save_users()`: При любой новой регистрации записывает изменения в файл.

Структура данных в JSON (как мы договорились):
```json
{
  "123456789": { "role": "student", "group": "МИСТ-25-1" },
  "987654321": { "role": "teacher", "name": "Иванов И.И." }
}

**Логика работы:**
1.  Загружаем данные через наш Парсер (один раз при старте!).
2.  При команде `/start` — показываем клавиатуру с группами.
3.  При выборе группы — запоминаем выбор в `users_db`.
4.  При запросе расписания — лезем в словарь с данными.

In [34]:
!pip install pyTelegramBotAPI python-dotenv --quiet


In [35]:
# Эмуляция создания файла .env
# В реальности вы создаете его руками: Файл -> Новый файл -> .env

env_content = """
BOT_TOKEN=5095342266:AAF5r5OgBWJmHm7mSuQW298mbvYrgw4KVlo
ADMIN_ID=73975940
"""

with open(".env", "w") as f:
    f.write(env_content)

In [36]:
import os
from dotenv import load_dotenv # Библиотека для чтения .env
import telebot
import logging # Взрослое логирование вместо print

# 1. Загружаем переменные из файла в окружение
load_dotenv()

# 2. Достаем токен
TOKEN = os.getenv('BOT_TOKEN')

# Проверка на дурака
if not TOKEN or TOKEN == 'ВАШ_ТОКЕН_ЗДЕСЬ_БЕЗ_КАВЫЧЕК':
    print("❌ ОШИБКА: Вы не вставили токен в файл .env (предыдущая ячейка)!")
else:
    print("✅ Токен найден. Продолжаем.")

# 3. Настройка логирования
# print() - это плохо, потому что не видно времени и уровня важности.
# logging - это стандарт.
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

# 4. Инициализация
bot = telebot.TeleBot(TOKEN)

✅ Токен найден. Продолжаем.


In [38]:
import os
import json
import logging
import telebot
from telebot import types
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Наш модуль парсера
import schedule_parser 

# ==========================================
# 1. КОНФИГУРАЦИЯ И ЛОГИРОВАНИЕ
# ==========================================
# Загружаем переменные окружения
load_dotenv()
TOKEN = os.getenv('BOT_TOKEN')

# Настраиваем логирование (Взрослый подход)
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

if not TOKEN:
    logger.error("Токен не найден! Создайте файл .env")
    exit()

bot = telebot.TeleBot(TOKEN)

# Файлы данных
SCHEDULE_FILE = 'МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx'
USERS_FILE = 'users.json'

# ==========================================
# 2. РАБОТА С ДАННЫМИ (Persistence Layer)
# ==========================================

# Глобальные переменные
schedule_data = {}
users_db = {} 

def load_data():
    """Загрузка всех данных при старте"""
    global schedule_data, users_db
    
    # 1. Парсим Excel
    logger.info("⏳ Загружаю расписание...")
    parser = schedule_parser.ScheduleParser(SCHEDULE_FILE)
    schedule_data = parser.parse()
    logger.info(f"✅ Расписание загружено. Групп: {len(schedule_data)}")

    # 2. Грузим пользователей из JSON
    if os.path.exists(USERS_FILE):
        try:
            with open(USERS_FILE, 'r', encoding='utf-8') as f:
                users_db = json.load(f)
            logger.info(f"✅ База пользователей загружена. Людей: {len(users_db)}")
        except Exception as e:
            logger.error(f"Ошибка чтения users.json: {e}")
            users_db = {}
    else:
        logger.warning("⚠️ Файл users.json не найден. Будет создан новый.")
        users_db = {}

def save_user(chat_id, role, data_value):
    """Сохраняет пользователя в память и в файл"""
    chat_id_str = str(chat_id) # JSON ключи должны быть строками
    
    users_db[chat_id_str] = {
        "role": role,
        "group": data_value if role == 'student' else None,
        "name": data_value if role == 'teacher' else None
    }
    
    try:
        with open(USERS_FILE, 'w', encoding='utf-8') as f:
            json.dump(users_db, f, ensure_ascii=False, indent=4)
        logger.info(f"💾 Пользователь {chat_id} сохранен ({role}: {data_value})")
    except Exception as e:
        logger.error(f"Ошибка записи в файл: {e}")

# Загружаем всё сразу при запуске скрипта
load_data()

# ==========================================
# 3. ИНТЕРФЕЙС БОТА (View Layer)
# ==========================================

@bot.message_handler(commands=['start'])
def start_command(message):
    logger.info(f"User {message.chat.id} pressed /start")
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True, row_width=2)
    
    # Кнопки групп
    buttons = [types.KeyboardButton(group) for group in schedule_data.keys()]
    markup.add(*buttons)
    
    # TODO: Сюда можно добавить кнопку "Я преподаватель" в будущем
    
    bot.send_message(
        message.chat.id, 
        "👋 **Привет! Я бот-староста.**\nВыберите свою группу:", 
        reply_markup=markup,
        parse_mode='Markdown'
    )

@bot.message_handler(func=lambda msg: msg.text in schedule_data.keys())
def register_student(message):
    """Регистрация студента"""
    group = message.text
    
    # Сохраняем в JSON
    save_user(message.chat.id, "student", group)
    
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("📅 На сегодня", "calendar На завтра")
    markup.row("🔙 Сменить группу")
    
    bot.send_message(
        message.chat.id, 
        f"✅ Вы записаны в **{group}**.\nДанные сохранены.", 
        reply_markup=markup, 
        parse_mode='Markdown'
    )

@bot.message_handler(func=lambda msg: msg.text in ["📅 На сегодня", " На завтра"])
def send_schedule(message):
    chat_id_str = str(message.chat.id)
    
    # Проверка регистрации (из JSON)
    if chat_id_str not in users_db:
        bot.send_message(message.chat.id, "⚠️ Сначала выберите группу через /start")
        return

    user_info = users_db[chat_id_str]
    group = user_info['group'] # Берем группу из профиля
    
    # Определяем дату
    target_date = datetime.now().date()
    if "На завтра" in message.text:
        target_date = target_date + timedelta(days=1)
    
    # Ищем пары
    lessons = [l for l in schedule_data.get(group, []) if l['start'].date() == target_date]
    
    if not lessons:
        bot.send_message(message.chat.id, f"🎉 Пар нет на {target_date.strftime('%d.%m')}!")
        return

    text = f"📅 **{target_date.strftime('%d.%m')} ({group})**:\n\n"
    for lesson in lessons:
        text += f"⏰ {lesson['start'].strftime('%H:%M')} - {lesson['end'].strftime('%H:%M')}\n"
        text += f"📚 {lesson['subject']}\n"
        # Используем данные, которые вытащил Regex
        text += f"👤 {lesson['teacher']}\n" 
        text += f"🚪 {lesson['room']}\n"
        text += "➖➖➖➖➖➖➖➖\n"

    bot.send_message(message.chat.id, text, parse_mode='Markdown')
    logger.info(f"Sent schedule to {message.chat.id} for {group}")

@bot.message_handler(func=lambda msg: msg.text == "🔙 Сменить группу")
def logout(message):
    start_command(message)

# ==========================================
# 4. ЗАПУСК
# ==========================================
if __name__ == '__main__':
    logger.info("🚀 Бот запускается...")
    bot.infinity_polling()

2025-12-01 23:57:42,001 - ⏳ Загружаю расписание...
2025-12-01 23:57:42,063 - ✅ Расписание загружено. Групп: 4
2025-12-01 23:57:42,064 - ⚠️ Файл users.json не найден. Будет создан новый.
2025-12-01 23:57:42,065 - 🚀 Бот запускается...


📂 Загружаю файл: МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx


2025-12-01 23:58:13,660 - User 73975940 pressed /start
2025-12-01 23:58:19,718 - 💾 Пользователь 73975940 сохранен (student: МИСТ-25-1-2)
2025-12-01 23:58:25,411 - Sent schedule to 73975940 for МИСТ-25-1-2
2025-12-01 23:58:27,950 - Sent schedule to 73975940 for МИСТ-25-1-2
2025-12-01 23:58:30,969 - User 73975940 pressed /start
2025-12-01 23:58:34,869 - 💾 Пользователь 73975940 сохранен (student: МИСТ-25-2-1)
2025-12-01 23:58:36,525 - Sent schedule to 73975940 for МИСТ-25-2-1
2025-12-01 23:59:13,989 (__init__.py:1121 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
2025-12-01 23:59:13,989 - Infinity polling: polling exited
2025-12-01 23:59:13,991 (__init__.py:1123 MainThread) ERROR - TeleBot: "Break infinity polling"
2025-12-01 23:59:13,991 - Break infinity polling


#  Часть 4.1: Ролевая модель. "А вы кто?"

Мы научили бота запоминать группу студента. Но мы забыли про преподавателей!
А ведь именно им должны приходить отчеты о прогульщиках.

**Задача:**
1.  При старте спрашивать: "Ты Студент или Преподаватель?".
2.  Если Преподаватель — дать ему выбрать свою Фамилию.
3.  **Сложность:** У нас нет списка преподавателей. У нас есть только расписание групп.

**Алгоритм (Data Engineering на лету):**
Нам нужно пройтись по всему словарю `schedule_data`, заглянуть в каждую пару, достать поле `teacher` и собрать **уникальный список** фамилий.

В Python для уникальности идеально подходит множество (`set`).

### 🛠 Обновляем логику загрузки данных

Нам нужно изменить функцию `load_data` в нашем боте. Теперь она будет не только парсить Excel, но и формировать список преподавателей.

Добавляем глобальную переменную `unique_teachers`.

In [39]:
import os
import json
import logging
import telebot
from telebot import types
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Подключаем наш парсер (убедитесь, что файл schedule_parser.py лежит рядом)
import schedule_parser 

# ==========================================
# 1. НАСТРОЙКИ И ЛОГИРОВАНИЕ
# ==========================================
load_dotenv()
TOKEN = os.getenv('BOT_TOKEN')

# Настраиваем красивый вывод логов
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

if not TOKEN:
    logger.error("❌ ОШИБКА: Токен не найден! Проверьте файл .env")
    exit()

bot = telebot.TeleBot(TOKEN)

# Файлы
SCHEDULE_FILE = 'МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx'
USERS_FILE = 'users.json'

# ==========================================
# 2. ДАННЫЕ (Глобальные переменные)
# ==========================================
schedule_data = {}    # Расписание: {'Группа': [уроки]}
users_db = {}         # Пользователи: {'chat_id': {'role':..., 'data':...}}
unique_teachers = []  # Список ФИО преподавателей

def load_data():
    """Загружает расписание, ищет преподавателей и читает базу пользователей"""
    global schedule_data, users_db, unique_teachers
    
    # --- Шаг 1: Парсинг Excel ---
    logger.info("⏳ Начинаю загрузку данных...")
    parser = schedule_parser.ScheduleParser(SCHEDULE_FILE)
    schedule_data = parser.parse()
    logger.info(f"✅ Расписание загружено. Найдено групп: {len(schedule_data)}")
    
    # --- Шаг 2: Поиск преподавателей (Data Mining) ---
    teachers_set = set() # Используем множество, чтобы убрать дубликаты
    
    for group, lessons in schedule_data.items():
        for lesson in lessons:
            t_name = lesson.get('teacher')
            # Фильтруем мусор и пустые значения
            if t_name and len(t_name) > 3 and t_name != 'Не указан':
                teachers_set.add(t_name)
    
    unique_teachers = sorted(list(teachers_set))
    logger.info(f"👨‍🏫 Из расписания извлечено преподавателей: {len(unique_teachers)}")

    # --- Шаг 3: Загрузка базы пользователей ---
    if os.path.exists(USERS_FILE):
        try:
            with open(USERS_FILE, 'r', encoding='utf-8') as f:
                users_db = json.load(f)
            logger.info(f"👥 База пользователей загружена: {len(users_db)} чел.")
        except Exception as e:
            logger.error(f"Ошибка чтения users.json: {e}")
            users_db = {}
    else:
        logger.warning("⚠️ Файл users.json не найден. Будет создан новый.")
        users_db = {}

def save_user(chat_id, role, info):
    """Сохраняет пользователя в файл"""
    chat_id_str = str(chat_id)
    
    # Формируем профиль пользователя
    users_db[chat_id_str] = {
        "role": role,          # 'student' или 'teacher'
        "info": info,          # Название группы или ФИО
        "registered_at": str(datetime.now())
    }
    
    # Записываем на диск
    try:
        with open(USERS_FILE, 'w', encoding='utf-8') as f:
            json.dump(users_db, f, ensure_ascii=False, indent=4)
        logger.info(f"💾 Пользователь {chat_id} сохранен как {role} ({info})")
    except Exception as e:
        logger.error(f"Не удалось сохранить пользователя: {e}")

# Загружаем данные при старте скрипта
load_data()

# ==========================================
# 3. ИНТЕРФЕЙС БОТА (Логика регистрации)
# ==========================================

@bot.message_handler(commands=['start'])
def start_command(message):
    """Главное меню: Выбор роли"""
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("🎓 Я Студент", "👨‍🏫 Я Преподаватель")
    
    bot.send_message(
        message.chat.id,
        "👋 **Привет! Я бот-староста.**\n\n"
        "Я умею показывать расписание и напоминать о парах.\n"
        "Для начала скажите: **кто вы?**",
        reply_markup=markup,
        parse_mode='Markdown'
    )

# --- ВЕТКА СТУДЕНТА ---

@bot.message_handler(func=lambda msg: msg.text == "🎓 Я Студент")
def student_reg_step1(message):
    """Показываем список групп"""
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True, row_width=2)
    # Генерируем кнопки из ключей словаря schedule_data
    btns = [types.KeyboardButton(g) for g in schedule_data.keys()]
    markup.add(*btns)
    markup.row("🔙 Назад")
    
    bot.send_message(message.chat.id, "Выберите вашу группу:", reply_markup=markup)

@bot.message_handler(func=lambda msg: msg.text in schedule_data.keys())
def student_reg_finish(message):
    """Финиш регистрации студента"""
    group = message.text
    save_user(message.chat.id, "student", group)
    
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("📅 Расписание на сегодня")
    markup.row("🔙 Сменить роль")
    
    bot.send_message(
        message.chat.id, 
        f"✅ Супер! Вы записаны в группу **{group}**.\nТеперь жмите кнопку, чтобы узнать пары.", 
        reply_markup=markup, parse_mode='Markdown'
    )

# --- ВЕТКА ПРЕПОДАВАТЕЛЯ ---

@bot.message_handler(func=lambda msg: msg.text == "👨‍🏫 Я Преподаватель")
def teacher_reg_step1(message):
    """Показываем список преподавателей"""
    if not unique_teachers:
        bot.send_message(message.chat.id, "⚠️ В расписании не найдено ни одного преподавателя.")
        return

    markup = types.ReplyKeyboardMarkup(resize_keyboard=True, row_width=1)
    # Генерируем кнопки из списка unique_teachers
    # (Берем первые 30, чтобы не заспамить экран, если их много)
    btns = [types.KeyboardButton(t) for t in unique_teachers[:30]]
    markup.add(*btns)
    markup.row("🔙 Назад")
    
    bot.send_message(
        message.chat.id, 
        "Найдите себя в списке (показаны первые 30):", 
        reply_markup=markup
    )

@bot.message_handler(func=lambda msg: msg.text in unique_teachers)
def teacher_reg_finish(message):
    """Финиш регистрации преподавателя"""
    name = message.text
    save_user(message.chat.id, "teacher", name)
    
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("📅 Мое расписание")
    markup.row("🔙 Сменить роль")
    
    bot.send_message(
        message.chat.id, 
        f"✅ Добрый день, **{name}**!\nТеперь вы будете видеть своё личное расписание.", 
        reply_markup=markup, parse_mode='Markdown'
    )

# --- ОБЩИЕ КНОПКИ ---

@bot.message_handler(func=lambda msg: msg.text in ["🔙 Назад", "🔙 Сменить роль"])
def back_handler(message):
    start_command(message)

# ==========================================
# 4. ЗАПУСК
# ==========================================
if __name__ == '__main__':
    logger.info("🚀 Бот запущен! Нажмите Ctrl+C для остановки.")
    bot.infinity_polling()

2025-12-02 00:09:02,363 - ⏳ Начинаю загрузку данных...
2025-12-02 00:09:02,431 - ✅ Расписание загружено. Найдено групп: 4
2025-12-02 00:09:02,432 - 👨‍🏫 Из расписания извлечено преподавателей: 7
2025-12-02 00:09:02,433 - 👥 База пользователей загружена: 1 чел.
2025-12-02 00:09:02,435 - 🚀 Бот запущен! Нажмите Ctrl+C для остановки.


📂 Загружаю файл: МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx


2025-12-02 00:09:12,447 (__init__.py:1115 MainThread) ERROR - TeleBot: "Infinity polling exception: HTTPSConnectionPool(host='api.telegram.org', port=443): Max retries exceeded with url: /bot5095342266:***********************************/getMe (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b867f515ee0>: Failed to resolve 'api.telegram.org' ([Errno -3] Temporary failure in name resolution)"))"
2025-12-02 00:09:12,447 - Infinity polling exception: HTTPSConnectionPool(host='api.telegram.org', port=443): Max retries exceeded with url: /bot5095342266:***********************************/getMe (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b867f515ee0>: Failed to resolve 'api.telegram.org' ([Errno -3] Temporary failure in name resolution)"))
2025-12-02 00:09:12,449 (__init__.py:1117 MainThread) ERROR - TeleBot: "Exception traceback:
Traceback (most recent call last):
  File "/home/daniel/misis/venv/lib/python3.12/site-package

#  Часть 4.2: Безопасность и Логика расписания

Мы нашли серьезную дыру в нашем алгоритме.
Сейчас любой студент может нажать "Я Преподаватель" и представиться кем угодно.

**Задача 1: Authentication (Проверка прав)**
Мы добавим "Гейткипер" (Gatekeeper).
1.  Пользователь жмет "Я Преподаватель".
2.  Бот не показывает список сразу, а спрашивает: **"Введите код доступа"**.
3.  Если код верный (например, `12345`) — пускаем дальше.
4.  Если нет — отправляем обратно.

В `telebot` для таких диалогов ("Вопрос-Ответ") используется метод `bot.register_next_step_handler`. Он говорит боту: *"Следующее сообщение от этого человека передай не в общий котел, а в конкретную функцию"*.

**Задача 2: Умный фильтр расписания**
Наша функция `send_schedule` сейчас глупая: она думает, что все люди — это студенты, и ищет пары только по Группе.
Нам нужно научить её:
* Если пишет Студент -> Искать по Группе.
* Если пишет Препод -> Искать по полю `teacher` в уроке.

In [40]:
import os
import json
import logging
import telebot
from telebot import types
from datetime import datetime, timedelta
from dotenv import load_dotenv

import schedule_parser 

# ==========================================
# 1. НАСТРОЙКИ
# ==========================================
load_dotenv()
TOKEN = os.getenv('BOT_TOKEN')
TEACHER_PASSWORD = "12345"  # 🔐 Пароль для преподавателей (в реале - в .env)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

if not TOKEN:
    exit("Error: BOT_TOKEN not found")

bot = telebot.TeleBot(TOKEN)

SCHEDULE_FILE = 'МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx'
USERS_FILE = 'users.json'

# ==========================================
# 2. ДАННЫЕ
# ==========================================
schedule_data = {}
users_db = {}
unique_teachers = []

def load_data():
    global schedule_data, users_db, unique_teachers
    logger.info("⏳ Загрузка данных...")
    
    # 1. Парсинг
    parser = schedule_parser.ScheduleParser(SCHEDULE_FILE)
    schedule_data = parser.parse()
    
    # 2. Поиск преподавателей
    teachers_set = set()
    for group, lessons in schedule_data.items():
        for lesson in lessons:
            t_name = lesson.get('teacher')
            if t_name and len(t_name) > 3 and t_name != 'Не указан':
                teachers_set.add(t_name)
    unique_teachers = sorted(list(teachers_set))
    logger.info(f"👨‍🏫 Преподавателей: {len(unique_teachers)}")

    # 3. Пользователи
    if os.path.exists(USERS_FILE):
        try:
            with open(USERS_FILE, 'r', encoding='utf-8') as f:
                users_db = json.load(f)
        except: users_db = {}
    else: users_db = {}

def save_user(chat_id, role, info):
    chat_id_str = str(chat_id)
    users_db[chat_id_str] = {"role": role, "info": info}
    try:
        with open(USERS_FILE, 'w', encoding='utf-8') as f:
            json.dump(users_db, f, ensure_ascii=False, indent=4)
    except Exception as e:
        logger.error(f"Save error: {e}")

load_data()

# ==========================================
# 3. ЛОГИКА РЕГИСТРАЦИИ
# ==========================================

@bot.message_handler(commands=['start'])
def start_command(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("🎓 Я Студент", "👨‍🏫 Я Преподаватель")
    bot.send_message(message.chat.id, "👋 Привет! Выберите роль:", reply_markup=markup)

# --- СТУДЕНТЫ ---
@bot.message_handler(func=lambda msg: msg.text == "🎓 Я Студент")
def reg_student(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True, row_width=2)
    markup.add(*[types.KeyboardButton(g) for g in schedule_data.keys()])
    bot.send_message(message.chat.id, "Выберите группу:", reply_markup=markup)

@bot.message_handler(func=lambda msg: msg.text in schedule_data.keys())
def reg_student_finish(message):
    save_user(message.chat.id, "student", message.text)
    send_main_menu(message.chat.id, "✅ Группа сохранена!")

# --- ПРЕПОДАВАТЕЛИ (С ЗАЩИТОЙ) ---
@bot.message_handler(func=lambda msg: msg.text == "👨‍🏫 Я Преподаватель")
def reg_teacher_start(message):
    # 🔐 ШАГ 1: Спрашиваем пароль
    msg = bot.send_message(message.chat.id, "🔒 Введите код доступа преподавателя:", reply_markup=types.ReplyKeyboardRemove())
    # Ждем следующего сообщения и направляем его в функцию check_password
    bot.register_next_step_handler(msg, check_teacher_password)

def check_teacher_password(message):
    # 🔐 ШАГ 2: Проверяем пароль
    if message.text == TEACHER_PASSWORD:
        markup = types.ReplyKeyboardMarkup(resize_keyboard=True, row_width=1)
        # Показываем список (ограничим 30, чтобы не спамить)
        markup.add(*[types.KeyboardButton(t) for t in unique_teachers[:30]])
        bot.send_message(message.chat.id, "🔓 Доступ разрешен!\nВыберите ваше ФИО:", reply_markup=markup)
        # Ждем выбора имени
        bot.register_next_step_handler(message, reg_teacher_finish)
    else:
        bot.send_message(message.chat.id, "⛔ Неверный пароль. Доступ запрещен.")
        start_command(message) # Возвращаем в начало

def reg_teacher_finish(message):
    if message.text in unique_teachers:
        save_user(message.chat.id, "teacher", message.text)
        send_main_menu(message.chat.id, f"✅ Здравствуйте, {message.text}!")
    else:
        bot.send_message(message.chat.id, "⚠️ Такого преподавателя нет в списке. Попробуйте еще раз.")
        start_command(message)

def send_main_menu(chat_id, text):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("📅 На сегодня", "calendar На завтра")
    markup.row("🔙 Сменить роль")
    bot.send_message(chat_id, text, reply_markup=markup)

# ==========================================
# 4. ЛОГИКА РАСПИСАНИЯ (УМНЫЙ ПОИСК)
# ==========================================

@bot.message_handler(func=lambda msg: msg.text in ["📅 На сегодня", "calendar На завтра"])
def get_schedule(message):
    chat_id_str = str(message.chat.id)
    if chat_id_str not in users_db:
        bot.send_message(message.chat.id, "Сначала зарегистрируйтесь через /start")
        return

    # 1. Определяем, кто спрашивает
    user = users_db[chat_id_str]
    role = user['role']     # 'student' или 'teacher'
    target = user['info']   # Название группы или ФИО препода
    
    # 2. Определяем дату
    search_date = datetime.now().date()
    if "На завтра" in message.text:
        search_date += timedelta(days=1)
    
    found_lessons = []

    # 3. Ищем пары (АЛГОРИТМ ЗАВИСИТ ОТ РОЛИ)
    if role == 'student':
        # Студент: просто берем расписание его группы
        group_lessons = schedule_data.get(target, [])
        found_lessons = [l for l in group_lessons if l['start'].date() == search_date]
        
    elif role == 'teacher':
        # Преподаватель: нужно перебрать ВСЕ группы и найти, где он ведет
        for group, lessons in schedule_data.items():
            for lesson in lessons:
                # Совпадает дата И совпадает фамилия
                if lesson['start'].date() == search_date and lesson['teacher'] == target:
                    # Добавляем поле группы, чтобы препод знал, к кому идти
                    lesson_copy = lesson.copy()
                    lesson_copy['group_name'] = group 
                    found_lessons.append(lesson_copy)
        
        # Сортируем пары преподавателя по времени
        found_lessons.sort(key=lambda x: x['start'])

    # 4. Вывод
    if not found_lessons:
        bot.send_message(message.chat.id, f"🎉 На {search_date.strftime('%d.%m')} пар нет!")
        return

    text = f"📅 **Расписание на {search_date.strftime('%d.%m')}**\n"
    text += f"👤 {role.capitalize()}: {target}\n\n"
    
    for lesson in found_lessons:
        start = lesson['start'].strftime('%H:%M')
        end = lesson['end'].strftime('%H:%M')
        
        text += f"⏰ `{start}-{end}`\n"
        text += f"📚 {lesson['subject']}\n"
        text += f"🚪 {lesson['room']}\n"
        
        # Если это препод, ему важно знать, КАКАЯ группа к нему придет
        if role == 'teacher':
             text += f"👥 Группа: **{lesson.get('group_name', 'Unknown')}**\n"
        else:
             text += f"👨‍🏫 {lesson['teacher']}\n"
             
        text += "➖" * 10 + "\n"

    bot.send_message(message.chat.id, text, parse_mode='Markdown')

@bot.message_handler(func=lambda msg: True)
def handle_text(message):
    if message.text == "🔙 Сменить роль":
        start_command(message)
    else:
        bot.send_message(message.chat.id, "Я не понимаю. Жмите кнопки 👇")

if __name__ == '__main__':
    logger.info("🚀 Бот запущен (v3.0 Secure)")
    bot.infinity_polling()

2025-12-02 00:13:59,776 - ⏳ Загрузка данных...
2025-12-02 00:13:59,846 - 👨‍🏫 Преподавателей: 7
2025-12-02 00:13:59,848 - 🚀 Бот запущен (v3.0 Secure)


📂 Загружаю файл: МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx


2025-12-02 00:14:00,099 (__init__.py:1115 MainThread) ERROR - TeleBot: "Infinity polling exception: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"
2025-12-02 00:14:00,099 - Infinity polling exception: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))
2025-12-02 00:14:00,102 (__init__.py:1117 MainThread) ERROR - TeleBot: "Exception traceback:
Traceback (most recent call last):
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
            

##  Разбор ключевых изменений

1.  **`bot.register_next_step_handler(msg, function)`**
    * Это мощный инструмент. Он ставит бота в режим "ожидания".
    * Бот запоминает, что следующее сообщение от этого юзера нужно отправить не в общий обработчик, а в функцию проверки пароля (`check_teacher_password`).

2.  **Поиск пар для Преподавателя:**
    * В блоке `elif role == 'teacher':` мы делаем перебор всех групп (`Nested Loops`).
    * Мы ищем совпадение `lesson['teacher'] == target` (ФИО преподавателя).
    * **Важный нюанс:** Мы добавляем поле `group_name` в найденный урок, чтобы преподаватель видел, к какой группе ему идти.

Теперь у нас есть защищенная система с разной логикой для разных пользователей. Запускайте и проверяйте пароль `12345`!

## Часть 4.3: Великий Планировщик и "Стукач"

Мы подошли к самой сложной части ТЗ.
Заказчик хочет:
1. **Напоминание:** За 1 час до пары прислать студентам кнопку "Я приду".
2. **Контроль:** Если через 30 минут кнопка не нажата — отправить список прогульщиков преподавателю.

### 💀 Почему наш старый подход здесь умрет?

Представьте, что мы пишем код так:

```python
# Псевдокод "Смерть Бота"
if time_to_notify:
    send_message("Нажми кнопку!")
    time.sleep(1800) # Ждем 30 минут (1800 секунд)
    check_who_pressed()
    send_report_to_teacher()
```

Как только программа дойдет до строчки `time.sleep(1800)`, ваш бот **уснет**.
В эти 30 минут он не будет отвечать на `/start`, не будет выдавать расписание. Он будет просто стоять. Это называется **Blocking I/O** (Блокирующая операция).

В реальной жизни официант, приняв у вас заказ, не стоит у вашего столика 20 минут, пока вы едите, игнорируя других гостей. Он идет обслуживать остальных и возвращается к вам, когда нужно.

Решение: Потоки (Threads)

Нам нужно разделить нашего бота на две независимые личности:
1. **Main Thread (Уши):** Слушает Телеграм и отвечает на команды мгновенно.
2. **Worker Thread (Будильник):** Висит в фоне, следит за временем и иногда дергает Главного: "Эй, пора делать рассылку!".



Для реализации нам понадобятся две библиотеки:
* `schedule` — чтобы красиво задавать время ("Каждую минуту", "Каждый день в 8:00").
* `threading` — чтобы запускать этот таймер параллельно с ботом.
"""


## Сборка конструктора (Main Bot)

Теперь у нас есть два мощных инструмента (Класса):
1. `DataManager` (знает всё о данных).
2. `AttendanceManager` (умеет следить за временем).

Файл `main_bot.py` — это **Связующее звено** (Controller). Он не содержит сложной бизнес-логики. Его задача — инициализировать объекты и перенаправлять сообщения от пользователей к нужному менеджеру.

Давайте разберем код построчно.
"""

### 1. Рождение Объектов (Instantiation)

Посмотрите на эти строки:
```python
data_mgr = DataManager('schedule.xlsx', 'users.json')
att_mgr = AttendanceManager(data_mgr, bot)
```

Здесь происходит магия ООП:
1. Мы создаем **экземпляр** класса `DataManager`. В этот момент срабатывает метод `__init__`, файл читается с диска и загружается в память. Переменная `data_mgr` — это наш "живой" библиотекарь.
2. Мы создаем `AttendanceManager`. Обратите внимание: мы передаем ему `data_mgr` внутрь!
   * *Зачем?* Потому что "Стукачу" (AttendanceManager) нужно знать список студентов, а этот список есть только у "Библиотекаря" (DataManager).
   * Это называется **Внедрение зависимости (Dependency Injection)**. Мы связываем объекты друг с другом.
"""

cell_main_threads = """
### 2. Запуск "Второго Мозга" (Threading)

```python
scheduler_thread = threading.Thread(target=att_mgr.run_scheduler_loop, daemon=True)
scheduler_thread.start()
```

* **Проблема:** Если мы запустим бесконечный цикл проверки времени (`while True`) прямо здесь, бот "зависнет" и перестанет отвечать на сообщения. Программа умеет делать только одну вещь за раз.
* **Решение:** Мы создаем отдельный **Поток (Thread)**.
    * Представьте, что `main` — это кассир, который общается с людьми.
    * А `scheduler_thread` — это повар на кухне, который следит за таймером духовки.
    * Они работают одновременно. `daemon=True` означает, что если мы убьем главного бота (кассира), повар тоже уйдет домой (поток закроется сам).
"""

### 3. Обработчики (Handlers) — Уши бота

Весь остальной код — это реакция на действия пользователя.
Заметьте, насколько чистым стал код функций!

**Пример регистрации студента:**
```python
@bot.message_handler(func=lambda m: m.text == "🎓 Студент")
def reg_stud(message):
    # Бот не лезет в Excel сам! Он просит DataManager.
    # "Эй, менеджер, дай мне ключи (названия групп) из расписания"
    for group in data_mgr.schedule.keys():
        ...
```

**Пример сохранения:**
```python
def reg_stud_finish(message):
    # Бот не работает с JSON сам!
    # Он делегирует эту задачу менеджеру.
    data_mgr.save_user(...)
```

**Итог:** Бот теперь работает как **Администратор**. Он принимает заявку ("Хочу зарегистрироваться"), передает её специалисту (`DataManager`), получает результат и отдает клиенту ("Готово").
"""

### 4. Инлайн-кнопки (Callback Query)

Когда студент нажимает кнопку "Я на месте", Телеграм отправляет специальный сигнал (не просто текст).

```python
@bot.callback_query_handler(...)
def checkin_handler(call):
    lesson_id = call.data.split('here_')[1]
    
    # Снова делегирование!
    # Всю сложную логику (проверку времени, удаление из списка прогульщиков)
    # делает AttendanceManager внутри метода process_student_click.
    success = att_mgr.process_student_click(...)
```

Если бы мы не использовали классы, здесь была бы "лапша" из 50 строк кода с проверками глобальных переменных. Сейчас — 3 строчки.
"""

In [10]:

import json
import os
import schedule_parser # Убедись, что файл schedule_parser.py лежит рядом

class DataManager:
    def __init__(self, schedule_file, users_file):
        self.schedule_file = schedule_file
        self.users_file = users_file
        
        # Данные храним внутри объекта
        self.schedule = {} 
        self.users = {}
        self.teachers_list = []
        
        # Загружаем сразу при создании
        self.load_all()

    def load_all(self):
        print("💾 [DataManager] Загружаю данные...")
        # 1. Excel
        parser = schedule_parser.ScheduleParser(self.schedule_file)
        self.schedule = parser.parse()
        
        # 2. Список преподавателей (генерируем из расписания)
        t_set = set()
        for lessons in self.schedule.values():
            for l in lessons:
                if l['teacher'] != 'Не указан':
                    t_set.add(l['teacher'])
        self.teachers_list = sorted(list(t_set))
        
        # 3. JSON Пользователей
        if os.path.exists(self.users_file):
            with open(self.users_file, 'r', encoding='utf-8') as f:
                self.users = json.load(f)
        else:
            self.users = {}

    def save_user(self, chat_id, role, info):
        self.users[str(chat_id)] = {
            "role": role, 
            "info": info, 
            "notifications": True
        }
        self._flush_users()

    def _flush_users(self):
        """Запись JSON на диск"""
        with open(self.users_file, 'w', encoding='utf-8') as f:
            json.dump(self.users, f, ensure_ascii=False, indent=4)

    def get_students_in_group(self, group_name):
        """Фильтр: вернуть ID всех студентов определенной группы"""
        return [uid for uid, data in self.users.items() 
                if data['role'] == 'student' and data['info'] == group_name]

    def get_teacher_id_by_name(self, name):
        """Поиск ID преподавателя по ФИО"""
        for uid, data in self.users.items():
            if data['role'] == 'teacher' and data['info'] == name:
                return uid
        return None

In [14]:
import threading
import time
import schedule
from datetime import datetime
from telebot import types

class AttendanceManager:
    def __init__(self, data_manager, bot):
        self.db = data_manager 
        self.bot = bot         
        self.active_checkins = {} 
        self.processed_lessons = set()

    def run_scheduler_loop(self):
        # Проверка каждые 10 секунд
        schedule.every(10).seconds.do(self._check_upcoming_lessons)
        while True:
            schedule.run_pending()
            time.sleep(1)

    def _check_upcoming_lessons(self):
        now = datetime.now()
        for group, lessons in self.db.schedule.items():
            for lesson in lessons:
                start = lesson['start']
                delta = (start - now).total_seconds() / 60
                
                lesson_id = f"{group}_{start.strftime('%Y%m%d%H%M')}"
                
                # Если осталось ~60 минут
                if 59 <= delta <= 61 and lesson_id not in self.processed_lessons:
                    self._start_checkin(group, lesson, lesson_id)

    def _start_checkin(self, group, lesson, lesson_id):
        self.processed_lessons.add(lesson_id)
        
        # --- 1. УВЕДОМЛЕНИЕ СТУДЕНТОВ ---
        students = self.db.get_students_in_group(group)
        if students:
            # Запись в память
            self.active_checkins[lesson_id] = {
                'teacher': lesson['teacher'],
                'subject': lesson['subject'],
                'absent': set(students)
            }
            
            markup = types.InlineKeyboardMarkup()
            btn = types.InlineKeyboardButton("✅ Я на месте", callback_data=f"here_{lesson_id}")
            markup.add(btn)
            
            print(f"📨 Рассылка студентам группы {group}...")
            for uid in students:
                try:
                    self.bot.send_message(uid, f"🔔 **Скоро пара!**\n{lesson['subject']}\n📍 {lesson['room']}", reply_markup=markup)
                except: pass
            
            # Таймер стукача
            delay = 30 # секунд для демо
            threading.Timer(delay, self._snitch_to_teacher, args=[lesson_id]).start()

        # --- 2. УВЕДОМЛЕНИЕ ПРЕПОДАВАТЕЛЯ (NEW) ---
        teacher_name = lesson['teacher']
        teacher_id = self.db.get_teacher_id_by_name(teacher_name)
        
        if teacher_id:
            print(f"📨 Напоминание преподавателю {teacher_name}...")
            text = (f"⏰ **Напоминание**\n"
                    f"Через час у вас пара: {lesson['subject']}\n"
                    f"Группа: {group}\n"
                    f"Аудитория: {lesson['room']}")
            try:
                self.bot.send_message(teacher_id, text, parse_mode='Markdown')
            except Exception as e:
                print(f"Ошибка отправки преподавателю: {e}")

    def process_student_click(self, lesson_id, user_id):
        user_id = str(user_id)
        if lesson_id in self.active_checkins:
            self.active_checkins[lesson_id]['absent'].discard(user_id)
            return True
        return False

    def _snitch_to_teacher(self, lesson_id):
        if lesson_id not in self.active_checkins: return
        
        data = self.active_checkins[lesson_id]
        absent_list = data['absent']
        
        if not absent_list: return

        teacher_uid = self.db.get_teacher_id_by_name(data['teacher'])
        
        if teacher_uid:
            text = (f"🚨 **Отчет о посещаемости**\n"
                    f"Пара: {data['subject']}\n"
                    f"Не отметились: {len(absent_list)} чел.")
            try:
                self.bot.send_message(teacher_uid, text, parse_mode='Markdown')
            except: pass
        
        del self.active_checkins[lesson_id]

In [7]:
import telebot
from telebot import types
import logging
from dotenv import load_dotenv

# --- Инициализация ---
load_dotenv() # Загружаем .env
TOKEN = os.getenv('BOT_TOKEN') 
if not TOKEN: raise ValueError("Нет токена!")

bot = telebot.TeleBot(TOKEN)

# 1. Создаем менеджеров
# Укажите точное имя вашего файла с расписанием!
data_mgr = DataManager('МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx', 'users.json')
att_mgr = AttendanceManager(data_mgr, bot)

# 2. Запускаем Поток Планировщика (Daemon=True значит, что он умрет вместе с основной программой)
scheduler_thread = threading.Thread(target=att_mgr.run_scheduler_loop, daemon=True)
scheduler_thread.start()
print("🚀 Планировщик запущен в фоне!")

# --- HANDLERS (Обработчики) ---

@bot.message_handler(commands=['start'])
def start(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("🎓 Студент", "👨‍🏫 Преподаватель")
    bot.send_message(message.chat.id, "Кто вы?", reply_markup=markup)

# --- Регистрация Студента ---
@bot.message_handler(func=lambda m: m.text == "🎓 Студент")
def reg_stud(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    # Берем группы из DataManager!
    for group in data_mgr.schedule.keys():
        markup.add(types.KeyboardButton(group))
    bot.send_message(message.chat.id, "Выберите группу:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in data_mgr.schedule.keys())
def reg_stud_finish(message):
    # Сохраняем через DataManager!
    data_mgr.save_user(message.chat.id, "student", message.text)
    bot.send_message(message.chat.id, "✅ Вы записаны! Теперь ждите уведомлений о парах.", reply_markup=types.ReplyKeyboardRemove())

# --- Регистрация Преподавателя ---
@bot.message_handler(func=lambda m: m.text == "👨‍🏫 Преподаватель")
def reg_teach(message):
    # Тут должна быть проверка пароля (как мы делали раньше), для краткости пропустим
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    # Берем список из DataManager
    for t in data_mgr.teachers_list[:30]: 
        markup.add(types.KeyboardButton(t))
    bot.send_message(message.chat.id, "Выберите себя:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in data_mgr.teachers_list)
def reg_teach_finish(message):
    data_mgr.save_user(message.chat.id, "teacher", message.text)
    bot.send_message(message.chat.id, "✅ Вы зарегистрированы как преподаватель. Вам будут приходить отчеты.", reply_markup=types.ReplyKeyboardRemove())

# --- ОБРАБОТКА КНОПКИ "Я НА МЕСТЕ" ---
@bot.callback_query_handler(func=lambda call: call.data.startswith('here_'))
def checkin_handler(call):
    # call.data выглядит как "here_МИСТ-25_202512010900"
    lesson_id = call.data.split('here_')[1]
    
    # Передаем обработку в AttendanceManager
    success = att_mgr.process_student_click(lesson_id, call.message.chat.id)
    
    if success:
        bot.answer_callback_query(call.id, "Отлично! Вы отмечены.")
        bot.edit_message_text("✅ Присутствие подтверждено.", call.message.chat.id, call.message.message_id)
    else:
        bot.answer_callback_query(call.id, "Либо время вышло, либо вы уже отметились.")

if __name__ == "__main__":
    print("🤖 Бот слушает...")
    bot.infinity_polling()

💾 [DataManager] Загружаю данные...
📂 [Parser] Читаю файл: МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx
✅ [Parser] Готово. Групп: 4
🚀 Планировщик запущен в фоне!
🤖 Бот слушает...


2025-12-02 10:03:32,541 (__init__.py:1121 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
2025-12-02 10:03:32,542 (__init__.py:1123 MainThread) ERROR - TeleBot: "Break infinity polling"


In [13]:
import telebot
from telebot import types
import threading
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Импортируем наши классы
# from data_manager import DataManager
# from attendance_manager import AttendanceManager

# --- Инициализация ---
load_dotenv()
TOKEN = os.getenv('BOT_TOKEN') 
if not TOKEN: raise ValueError("Нет токена!")

TEACHER_PASSWORD = "12345" # Пароль для преподавателей

bot = telebot.TeleBot(TOKEN)

# 1. Создаем менеджеров
data_mgr = DataManager('МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx', 'users.json')
att_mgr = AttendanceManager(data_mgr, bot)

# 2. Запускаем Поток Планировщика
scheduler_thread = threading.Thread(target=att_mgr.run_scheduler_loop, daemon=True)
scheduler_thread.start()
print("🚀 Бот и Планировщик запущены!")

# ==========================================
# 🎮 ИНТЕРФЕЙС (VIEW LAYER)
# ==========================================

@bot.message_handler(commands=['start'])
def start(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("🎓 Студент", "👨‍🏫 Преподаватель")
    bot.send_message(message.chat.id, "👋 Привет! Кто вы?", reply_markup=markup)

# --- ЛОГИКА СТУДЕНТА ---
@bot.message_handler(func=lambda m: m.text == "🎓 Студент")
def reg_stud(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    # Берем группы напрямую у менеджера данных
    for group in data_mgr.schedule.keys():
        markup.add(types.KeyboardButton(group))
    markup.row("🔙 Назад")
    bot.send_message(message.chat.id, "Выберите вашу группу:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in data_mgr.schedule.keys())
def reg_stud_finish(message):
    data_mgr.save_user(message.chat.id, "student", message.text)
    send_main_menu(message.chat.id, f"✅ Вы записаны в группу **{message.text}**!")

# --- ЛОГИКА ПРЕПОДАВАТЕЛЯ (С ЗАЩИТОЙ) ---
@bot.message_handler(func=lambda m: m.text == "👨‍🏫 Преподаватель")
def reg_teach(message):
    msg = bot.send_message(message.chat.id, "🔒 Введите код доступа:", reply_markup=types.ReplyKeyboardRemove())
    # Ждем следующего сообщения (пароль)
    bot.register_next_step_handler(msg, check_teacher_pass)

def check_teacher_pass(message):
    if message.text == TEACHER_PASSWORD:
        markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
        # Берем список преподавателей у менеджера
        for t in data_mgr.teachers_list[:30]: 
            markup.add(types.KeyboardButton(t))
        bot.send_message(message.chat.id, "🔓 Доступ разрешен. Выберите ФИО:", reply_markup=markup)
        bot.register_next_step_handler(message, reg_teach_finish)
    else:
        bot.send_message(message.chat.id, "⛔ Неверный пароль.")
        start(message)

def reg_teach_finish(message):
    if message.text in data_mgr.teachers_list:
        data_mgr.save_user(message.chat.id, "teacher", message.text)
        send_main_menu(message.chat.id, f"✅ Здравствуйте, **{message.text}**!")
    else:
        bot.send_message(message.chat.id, "Такого преподавателя нет в списке.")
        start(message)

# --- ОБЩЕЕ МЕНЮ И РАСПИСАНИЕ ---
def send_main_menu(chat_id, text):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("📅 На сегодня", "📅 На завтра")
    markup.row("🔙 Сменить роль")
    bot.send_message(chat_id, text, reply_markup=markup, parse_mode='Markdown')

@bot.message_handler(func=lambda m: m.text in ["📅 На сегодня", "📅 На завтра"])
def get_schedule(message):
    # 1. Получаем пользователя через Менеджер
    user = data_mgr.users.get(str(message.chat.id))
    if not user:
        bot.send_message(message.chat.id, "⚠️ Сначала зарегистрируйтесь через /start")
        return

    # 2. Определяем дату
    target_date = datetime.now().date()
    if "На завтра" in message.text:
        target_date += timedelta(days=1)

    # 3. Ищем пары (Логика зависит от роли)
    found_lessons = []
    role = user['role']
    target = user['info'] # Название группы или ФИО

    if role == 'student':
        # Студент: просто берем расписание группы из словаря
        lessons = data_mgr.schedule.get(target, [])
        found_lessons = [l for l in lessons if l['start'].date() == target_date]
    
    elif role == 'teacher':
        # Преподаватель: перебор всех групп
        for group_name, lessons in data_mgr.schedule.items():
            for l in lessons:
                if l['start'].date() == target_date and l['teacher'] == target:
                    # Важно: Копируем и добавляем имя группы, чтобы препод знал, к кому идти
                    l_copy = l.copy()
                    l_copy['group_name'] = group_name
                    found_lessons.append(l_copy)
        found_lessons.sort(key=lambda x: x['start'])

    # 4. Вывод
    if not found_lessons:
        bot.send_message(message.chat.id, "🎉 Пар нет!")
        return

    text = f"📅 **Расписание на {target_date.strftime('%d.%m')}**\n\n"
    for l in found_lessons:
        text += f"⏰ {l['start'].strftime('%H:%M')} - {l['end'].strftime('%H:%M')}\n"
        text += f"📚 {l['subject']}\n"
        text += f"🚪 {l['room']}\n"
        
        if role == 'teacher':
            text += f"👥 Группа: **{l['group_name']}**\n"
        else:
            text += f"👨‍🏫 {l['teacher']}\n"
        text += "➖➖➖➖➖➖➖\n"

    bot.send_message(message.chat.id, text, parse_mode='Markdown')

# --- КНОПКА "НАЗАД" ---
@bot.message_handler(func=lambda m: m.text in ["🔙 Назад", "🔙 Сменить роль"])
def back(message):
    start(message)

# --- CALLBACK ОТМЕТКИ (Делегируем AttendanceManager) ---
@bot.callback_query_handler(func=lambda call: call.data.startswith('here_'))
def checkin_handler(call):
    lesson_id = call.data.split('here_')[1]
    success = att_mgr.process_student_click(lesson_id, call.message.chat.id)
    
    if success:
        bot.answer_callback_query(call.id, "Вы отмечены! ✅")
        bot.edit_message_text("✅ Присутствие подтверждено.", call.message.chat.id, call.message.message_id)
    else:
        bot.answer_callback_query(call.id, "Ошибка или время вышло 🚫")

if __name__ == "__main__":
    bot.infinity_polling()

💾 [DataManager] Загружаю данные...
📂 [Parser] Читаю файл: МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx
✅ [Parser] Готово. Групп: 4
🚀 Бот и Планировщик запущены!


2025-12-02 10:06:55,698 (__init__.py:1115 MainThread) ERROR - TeleBot: "Infinity polling exception: HTTPSConnectionPool(host='api.telegram.org', port=443): Max retries exceeded with url: /bot5095342266:***********************************/sendMessage?chat_id=73975940&text=%F0%9F%91%8B+%D0%9F%D1%80%D0%B8%D0%B2%D0%B5%D1%82%21+%D0%9A%D1%82%D0%BE+%D0%B2%D1%8B%3F&reply_markup=%7B%22keyboard%22%3A+%5B%5B%7B%22text%22%3A+%22%5Cud83c%5Cudf93+%5Cu0421%5Cu0442%5Cu0443%5Cu0434%5Cu0435%5Cu043d%5Cu0442%22%7D%2C+%7B%22text%22%3A+%22%5Cud83d%5Cudc68%5Cu200d%5Cud83c%5Cudfeb+%5Cu041f%5Cu0440%5Cu0435%5Cu043f%5Cu043e%5Cu0434%5Cu0430%5Cu0432%5Cu0430%5Cu0442%5Cu0435%5Cu043b%5Cu044c%22%7D%5D%5D%2C+%22resize_keyboard%22%3A+true%7D (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b11286e7f80>: Failed to resolve 'api.telegram.org' ([Errno -3] Temporary failure in name resolution)"))"
2025-12-02 10:06:55,701 (__init__.py:1117 MainThread) ERROR - TeleBot: "Exception traceback:
Trac

## Часть 4.7: "А про преподавателя забыли?"

Мы сделали рассылку студентам, но преподаватель тоже человек! Ему тоже нужно напомнить, что через час у него пара, и подсказать аудиторию (чтобы он не искал её в расписании).

**Задача:**
Добавить в `AttendanceManager` логику:
1. Когда запускается проверка (`_start_checkin`), достать имя преподавателя из урока.
2. Найти его `chat_id` в базе.
3. Отправить ему персональное уведомление (без кнопок, просто текст).

Давайте обновим наш класс `AttendanceManager`.

## Часть 5: Как это тестировать? (Debug Mode)

Мы написали кучу кода. Но как проверить, что уведомления приходят?
Ждать, пока наступит реальное время пары (например, следующий вторник) — плохая идея для лекции.

Программисты используют **Mocking** (Имитацию) или ручной вызов функций.

Мы сделаем "Кнопку Судного Дня" (или команду /test), которая:

1. Ищет в базе любую группу, у которой ведет Воробьев Даниил Анатольевич.

2. "Обманывает" систему, говоря, что пара у этой группы ровно через 1 час.

3. Запускает рассылку студентам.

4. Ставит таймер "Стукача" (отчета) ровно на 1 минуту (60 секунд).



Нам понадобятся изменения в двух файлах.

1. AttendanceManager (Добавляем тестовый сценарий)
Мы немного улучшим метод _start_checkin, чтобы он принимал аргумент snitch_delay (через сколько секунд жаловаться). И добавим новый метод run_test_scenario.

2. main_bot.py (Добавляем команду /test)
Добавим этот обработчик в файл основного бота. Когда вы напишете /test в чат, запустится сценарий.



In [22]:
import threading
import time
import schedule
from datetime import datetime, timedelta
from telebot import types

class AttendanceManager:
    def __init__(self, data_manager, bot):
        self.db = data_manager 
        self.bot = bot         
        self.active_checkins = {} 
        self.processed_lessons = set()

    def run_scheduler_loop(self):
        # Обычная проверка раз в минуту
        schedule.every(1).minutes.do(self._check_upcoming_lessons)
        while True:
            schedule.run_pending()
            time.sleep(1)

    def _check_upcoming_lessons(self):
        # Штатный режим: ищем пары через 1 час
        self.manual_trigger(hours=1, is_auto=True)

    def manual_trigger(self, hours, is_auto=False):
        """Поиск реальных пар в Excel"""
        now = datetime.now()
        triggered = 0
        
        print(f"🔎 Проверка пар через {hours} ч...")

        for group, lessons in self.db.schedule.items():
            for lesson in lessons:
                start = lesson['start']
                diff = (start - now).total_seconds() / 3600
                
                # Попали в окно времени (±30 мин)
                if abs(diff - hours) <= 0.5:
                    suffix = "" if is_auto else f"_MAN_{datetime.now().strftime('%M%S')}"
                    lesson_id = f"{group}_{start.strftime('%Y%m%d%H%M')}{suffix}"
                    
                    if lesson_id not in self.processed_lessons:
                        # В штатном режиме стукач срабатывает через 30 минут (1800 сек)
                        self._start_checkin(group, lesson, lesson_id, hours, snitch_delay=1800)
                        triggered += 1
        return triggered

    # --- 🧪 СИМУЛЯТОР ДЛЯ ЛЕКЦИИ ---
    def run_test_scenario(self, target_teacher="Воробьев Даниил Анатольевич"):
        """
        Создает ФЕЙКОВУЮ пару ровно через 1 час от текущего момента.
        Гарантирует срабатывание уведомлений.
        """
        print(f"🧪 ЗАПУСК СИМУЛЯЦИИ для: {target_teacher}")
        
        now = datetime.now()
        fake_start = now + timedelta(hours=1) # Пара через час
        
        # 1. Создаем виртуальный урок
        fake_lesson = {
            'subject': '🕹️ Тестовая Лекция (Simulation)',
            'teacher': target_teacher,
            'start': fake_start,
            'end': fake_start + timedelta(minutes=90),
            'room': 'Виртуальная 404'
        }
        
        # 2. Ищем любую группу, чтобы было кому отправить
        # (Берем первую попавшуюся из базы, где есть студенты)
        target_group = None
        for group_name in self.db.schedule.keys():
            if self.db.get_students_in_group(group_name):
                target_group = group_name
                break
        
        if not target_group:
            return "❌ Ошибка: В базе нет студентов ни в одной группе."

        # 3. Генерируем уникальный ID
        test_id = f"TEST_{now.strftime('%H%M%S')}"
        
        # 4. Запускаем! Таймер стукача ставим на 60 СЕКУНД (чтобы быстро показать)
        self._start_checkin(target_group, fake_lesson, test_id, hours_left=1, snitch_delay=60)
        
        return f"✅ Тест запущен!\nГруппа: {target_group}\nПредмет: {fake_lesson['subject']}\n⏳ 'Стукач' сработает через 60 сек!"

    def _start_checkin(self, group, lesson, lesson_id, hours_left, snitch_delay=1800):
        self.processed_lessons.add(lesson_id)
        
        time_hint = f"через {hours_left} ч." if hours_left >= 1 else "скоро"
        if hours_left == 1: time_hint = "через 1 час"

        # --- 1. СТУДЕНТЫ ---
        students = self.db.get_students_in_group(group)
        if students:
            # Запоминаем прогульщиков
            self.active_checkins[lesson_id] = {
                'teacher': lesson['teacher'],
                'subject': lesson['subject'],
                'absent': set(students)
            }
            
            markup = types.InlineKeyboardMarkup()
            markup.add(types.InlineKeyboardButton("✅ Я буду", callback_data=f"here_{lesson_id}"))
            
            print(f"📨 Рассылка студентам {group}...")
            for uid in students:
                try:
                    self.bot.send_message(
                        uid, 
                        f"🔔 **НАПОМИНАНИЕ!**\n(Пара {time_hint})\n\n📚 {lesson['subject']}\n📍 {lesson['room']}", 
                        reply_markup=markup,
                        parse_mode='Markdown'
                    )
                except: pass
            
            # ЗАПУСК ТАЙМЕРА (Стукач)
            print(f"⏱ Таймер отчета взведен на {snitch_delay} сек.")
            threading.Timer(snitch_delay, self._snitch_to_teacher, args=[lesson_id]).start()

        # --- 2. ПРЕПОДАВАТЕЛЬ ---
        teacher_id = self.db.get_teacher_id_by_name(lesson['teacher'])
        if teacher_id:
            try:
                self.bot.send_message(
                    teacher_id,
                    f"⏰ **Напоминание преподавателю**\nПара {time_hint}: {lesson['subject']}\n"
                    f"👥 Группа: {group}\n🚪 Аудитория: {lesson['room']}\n\n"
                    f"ℹ️ *Контроль присутствия запущен.*",
                    parse_mode='Markdown'
                )
            except: pass

    def process_student_click(self, lesson_id, user_id):
        user_id = str(user_id)
        if lesson_id in self.active_checkins:
            self.active_checkins[lesson_id]['absent'].discard(user_id)
            return True
        return False

    def _snitch_to_teacher(self, lesson_id):
        print(f"📝 Формирование отчета для {lesson_id}")
        if lesson_id not in self.active_checkins: return
        
        data = self.active_checkins[lesson_id]
        absent = data['absent']
        
        # Если список пуст - не спамим
        if not absent: 
            print("Все пришли.")
            del self.active_checkins[lesson_id]
            return
        
        t_id = self.db.get_teacher_id_by_name(data['teacher'])
        if t_id:
            try:
                self.bot.send_message(
                    t_id, 
                    f"🚨 **Отчет о прогульщиках**\n"
                    f"Предмет: {data['subject']}\n"
                    f"Не нажали кнопку: {len(absent)} чел.\n"
                    f"(ID: {', '.join(list(absent))})",
                    parse_mode='Markdown'
                )
                print("✅ Отчет отправлен.")
            except: pass
            
        del self.active_checkins[lesson_id]

In [24]:
import telebot
from telebot import types
import threading
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

# from data_manager import DataManager
# from attendance_manager import AttendanceManager

load_dotenv()
TOKEN = os.getenv('BOT_TOKEN') 
if not TOKEN: raise ValueError("Нет токена!")

TEACHER_PASSWORD = "12345"

bot = telebot.TeleBot(TOKEN)

# Инициализация
data_mgr = DataManager('МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx', 'users.json')
att_mgr = AttendanceManager(data_mgr, bot)

# Запуск планировщика
threading.Thread(target=att_mgr.run_scheduler_loop, daemon=True).start()
print("🚀 Бот запущен!")

# --- 🧪 ТЕСТОВАЯ КНОПКА (СИМУЛЯТОР) ---
@bot.message_handler(commands=['test'])
def run_simulation(message):
    """
    Запускает сценарий:
    1. Создает фейковую пару "через час".
    2. Рассылает уведомления.
    3. Присылает отчет через 1 минуту.
    """
    bot.send_message(message.chat.id, "🚀 Запускаю симуляцию лекции...")
    
    # Запускаем тест для конкретного преподавателя
    # Убедитесь, что вы зарегистрировались под этим именем, чтобы получить отчет!
    result = att_mgr.run_test_scenario("Воробьев Даниил Анатольевич")
    
    bot.send_message(message.chat.id, result)

# --- АДМИН-КОМАНДА (Ручной поиск) ---
@bot.message_handler(commands=['notify'])
def manual_notify(message):
    try:
        parts = message.text.split()
        hours = float(parts[1]) if len(parts) > 1 else 1.0
        count = att_mgr.manual_trigger(hours)
        bot.reply_to(message, f"📣 Проверка завершена. Найдено пар: {count}")
    except ValueError:
        bot.reply_to(message, "Ошибка. Пишите: `/notify 2`")

# --- СТАНДАРТНЫЙ ИНТЕРФЕЙС ---

@bot.message_handler(commands=['start'])
def start(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("🎓 Студент", "👨‍🏫 Преподаватель")
    bot.send_message(message.chat.id, "👋 Выберите роль:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text == "🎓 Студент")
def reg_stud(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    for group in data_mgr.schedule.keys():
        markup.add(types.KeyboardButton(group))
    markup.row("🔙 Назад")
    bot.send_message(message.chat.id, "Выберите группу:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in data_mgr.schedule.keys())
def reg_stud_finish(message):
    data_mgr.save_user(message.chat.id, "student", message.text)
    send_menu(message.chat.id)

@bot.message_handler(func=lambda m: m.text == "👨‍🏫 Преподаватель")
def reg_teach(message):
    msg = bot.send_message(message.chat.id, "🔒 Введите код:", reply_markup=types.ReplyKeyboardRemove())
    bot.register_next_step_handler(msg, check_pass)

def check_pass(message):
    if message.text == TEACHER_PASSWORD:
        markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
        for t in data_mgr.teachers_list[:30]: markup.add(types.KeyboardButton(t))
        bot.send_message(message.chat.id, "Выберите ФИО:", reply_markup=markup)
        bot.register_next_step_handler(message, reg_teach_finish)
    else:
        bot.send_message(message.chat.id, "⛔ Неверно.")
        start(message)

def reg_teach_finish(message):
    if message.text in data_mgr.teachers_list:
        data_mgr.save_user(message.chat.id, "teacher", message.text)
        send_menu(message.chat.id)
    else:
        bot.send_message(message.chat.id, "Нет в списке.")
        start(message)

def send_menu(chat_id):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("📅 Сегодня", "📅 Завтра", "🔙 Назад")
    bot.send_message(chat_id, "✅ Вы в системе!", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in ["📅 Сегодня", "📅 Завтра"])
def get_sched(message):
    user = data_mgr.users.get(str(message.chat.id))
    if not user: return bot.send_message(message.chat.id, "Сначала /start")
    
    target_date = datetime.now().date()
    if "Завтра" in message.text: target_date += timedelta(days=1)
    
    found = []
    if user['role'] == 'student':
        lessons = data_mgr.schedule.get(user['info'], [])
        found = [l for l in lessons if l['start'].date() == target_date]
    elif user['role'] == 'teacher':
        for gr, lessons in data_mgr.schedule.items():
            for l in lessons:
                if l['start'].date() == target_date and l['teacher'] == user['info']:
                    l_copy = l.copy()
                    l_copy['group'] = gr
                    found.append(l_copy)
        found.sort(key=lambda x: x['start'])
        
    if not found: return bot.send_message(message.chat.id, "Пар нет!")
    
    text = f"📅 **{target_date.strftime('%d.%m')}**\n\n"
    for l in found:
        text += f"⏰ {l['start'].strftime('%H:%M')}\n📚 {l['subject']}\n🚪 {l['room']}\n"
        if user['role'] == 'teacher': text += f"👥 {l['group']}\n"
        else: text += f"👨‍🏫 {l['teacher']}\n"
        text += "➖\n"
    bot.send_message(message.chat.id, text, parse_mode='Markdown')

@bot.message_handler(func=lambda m: m.text == "🔙 Назад")
def back(m): start(m)

@bot.callback_query_handler(func=lambda call: call.data.startswith('here_'))
def checkin(call):
    lid = call.data.split('here_')[1]
    if att_mgr.process_student_click(lid, call.message.chat.id):
        bot.edit_message_text("✅ Вы отмечены!", call.message.chat.id, call.message.message_id)
    else:
        bot.answer_callback_query(call.id, "Ошибка")

if __name__ == "__main__":
    bot.infinity_polling()

💾 [DataManager] Загружаю данные...
📂 [Parser] Читаю файл: МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx
✅ [Parser] Готово. Групп: 4
🚀 Бот запущен!
🧪 ЗАПУСК СИМУЛЯЦИИ для: Воробьев Даниил Анатольевич
📨 Рассылка студентам МИСТ-25-2-1...
⏱ Таймер отчета взведен на 60 сек.
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
📝 Формирование отчета для TEST_103558
Все пришли.
🔎 Проверка пар через 1 ч...
🧪 ЗАПУСК СИМУЛЯЦИИ для: Воробьев Даниил Анатольевич
📨 Рассылка студентам МИСТ-25-2-1...
⏱ Таймер отчета взведен на 60 сек.
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
📝 Формирование отчета для TEST_103740
✅ Отчет отправлен.
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч...
🔎 Проверка пар через 1 ч

2025-12-02 10:47:02,708 (__init__.py:1115 MainThread) ERROR - TeleBot: "Infinity polling exception: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))"
2025-12-02 10:47:02,711 (__init__.py:1117 MainThread) ERROR - TeleBot: "Exception traceback:
Traceback (most recent call last):
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 787, in urlopen
    response = self._make_request(
               ^^^^^^^^^^^^^^^^^^^
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/http/client.py", line 1428, in getresponse
    response.begin()
  File "/usr/lib/pytho

🔎 Проверка пар через 1 ч...


2025-12-02 10:47:39,292 (__init__.py:1115 MainThread) ERROR - TeleBot: "Infinity polling exception: HTTPSConnectionPool(host='api.telegram.org', port=443): Max retries exceeded with url: /bot5095342266:***********************************/getUpdates?offset=430509702&timeout=20 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b11284c14f0>: Failed to resolve 'api.telegram.org' ([Errno -3] Temporary failure in name resolution)"))"
2025-12-02 10:47:39,294 (__init__.py:1117 MainThread) ERROR - TeleBot: "Exception traceback:
Traceback (most recent call last):
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^

🔎 Проверка пар через 1 ч...


...

2025-12-02 12:11:51,123 (__init__.py:1115 MainThread) ERROR - TeleBot: "Infinity polling exception: HTTPSConnectionPool(host='api.telegram.org', port=443): Max retries exceeded with url: /bot5095342266:***********************************/getUpdates?offset=430509702&timeout=20 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b11284bfa10>: Failed to resolve 'api.telegram.org' ([Errno -3] Temporary failure in name resolution)"))"
2025-12-02 12:11:51,125 (__init__.py:1117 MainThread) ERROR - TeleBot: "Exception traceback:
Traceback (most recent call last):
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^

🔎 Проверка пар через 1 ч...


2025-12-02 12:12:15,990 (__init__.py:1115 MainThread) ERROR - TeleBot: "Infinity polling exception: HTTPSConnectionPool(host='api.telegram.org', port=443): Max retries exceeded with url: /bot5095342266:***********************************/getUpdates?offset=430509702&timeout=20 (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7b1128346a50>: Failed to resolve 'api.telegram.org' ([Errno -3] Temporary failure in name resolution)"))"
2025-12-02 12:12:15,992 (__init__.py:1117 MainThread) ERROR - TeleBot: "Exception traceback:
Traceback (most recent call last):
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/daniel/misis/venv/lib/python3.12/site-packages/urllib3/util/connection.py", line 60, in create_connection
    for res in socket.getaddrinfo(host, port, family, socket.SOCK_STREAM):
               ^^^^^^^^^

🔎 Проверка пар через 1 ч...


2025-12-02 12:12:48,428 (__init__.py:1121 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
2025-12-02 12:12:48,429 (__init__.py:1123 MainThread) ERROR - TeleBot: "Break infinity polling"


# Вернемся сюда после части про WEB

In [ ]:
import telebot
from telebot import types
import threading
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

from data_manager import DataManager
from attendance_manager import AttendanceManager

load_dotenv()
TOKEN = os.getenv('BOT_TOKEN') 
if not TOKEN: raise ValueError("Нет токена!")

TEACHER_PASSWORD = "12345"

bot = telebot.TeleBot(TOKEN)

# --- ГЛАВНОЕ ИЗМЕНЕНИЕ ЗДЕСЬ ---
# Мы указываем путь к файлу, который загружает Flask ('data/uploaded_schedule.xlsx')
# Если файла еще нет (админ не загрузил), используем запасной ('fallback.xlsx' или старый путь)

schedule_path = 'data/uploaded_schedule.xlsx'
if not os.path.exists(schedule_path):
    print(f"⚠️ Внимание: Файл {schedule_path} не найден! Использую старый файл.")
    schedule_path = 'МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx'

data_mgr = DataManager(schedule_path, 'users.json')
att_mgr = AttendanceManager(data_mgr, bot)

# Запуск планировщика
threading.Thread(target=att_mgr.run_scheduler_loop, daemon=True).start()
print(f"🚀 Бот запущен! Читаю расписание из: {schedule_path}")

# --- 🧪 ТЕСТОВАЯ КНОПКА (СИМУЛЯТОР) ---
@bot.message_handler(commands=['test'])
def run_simulation(message):
    bot.send_message(message.chat.id, "🚀 Запускаю симуляцию лекции...")
    # Убедитесь, что имя преподавателя совпадает с тем, что в новом файле!
    result = att_mgr.run_test_scenario("Воробьев Даниил Анатольевич")
    bot.send_message(message.chat.id, result)

# --- АДМИН-КОМАНДА /notify ---
@bot.message_handler(commands=['notify'])
def manual_notify(message):
    try:
        parts = message.text.split()
        hours = float(parts[1]) if len(parts) > 1 else 1.0
        count = att_mgr.manual_trigger(hours)
        bot.reply_to(message, f"📣 Проверка завершена. Найдено пар: {count}")
    except ValueError:
        bot.reply_to(message, "Ошибка. Пишите: `/notify 2`")

# --- СТАНДАРТНЫЙ ИНТЕРФЕЙС ---

@bot.message_handler(commands=['start'])
def start(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("🎓 Студент", "👨‍🏫 Преподаватель")
    bot.send_message(message.chat.id, "👋 Выберите роль:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text == "🎓 Студент")
def reg_stud(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    for group in data_mgr.schedule.keys():
        markup.add(types.KeyboardButton(group))
    markup.row("🔙 Назад")
    bot.send_message(message.chat.id, "Выберите группу:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in data_mgr.schedule.keys())
def reg_stud_finish(message):
    data_mgr.save_user(message.chat.id, "student", message.text)
    send_menu(message.chat.id)

@bot.message_handler(func=lambda m: m.text == "👨‍🏫 Преподаватель")
def reg_teach(message):
    msg = bot.send_message(message.chat.id, "🔒 Введите код:", reply_markup=types.ReplyKeyboardRemove())
    bot.register_next_step_handler(msg, check_pass)

def check_pass(message):
    if message.text == TEACHER_PASSWORD:
        markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
        # Показываем первые 30 преподавателей
        for t in data_mgr.teachers_list[:30]: markup.add(types.KeyboardButton(t))
        bot.send_message(message.chat.id, "Выберите ФИО:", reply_markup=markup)
        bot.register_next_step_handler(message, reg_teach_finish)
    else:
        bot.send_message(message.chat.id, "⛔ Неверно.")
        start(message)

def reg_teach_finish(message):
    if message.text in data_mgr.teachers_list:
        data_mgr.save_user(message.chat.id, "teacher", message.text)
        send_menu(message.chat.id)
    else:
        bot.send_message(message.chat.id, "Нет в списке.")
        start(message)

def send_menu(chat_id):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("📅 Сегодня", "📅 Завтра", "🔙 Назад")
    bot.send_message(chat_id, "✅ Вы в системе!", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in ["📅 Сегодня", "📅 Завтра"])
def get_sched(message):
    user = data_mgr.users.get(str(message.chat.id))
    if not user: return bot.send_message(message.chat.id, "Сначала /start")
    
    target_date = datetime.now().date()
    if "Завтра" in message.text: target_date += timedelta(days=1)
    
    found = []
    if user['role'] == 'student':
        lessons = data_mgr.schedule.get(user['info'], [])
        found = [l for l in lessons if l['start'].date() == target_date]
    elif user['role'] == 'teacher':
        for gr, lessons in data_mgr.schedule.items():
            for l in lessons:
                if l['start'].date() == target_date and l['teacher'] == user['info']:
                    l_copy = l.copy()
                    l_copy['group'] = gr
                    found.append(l_copy)
        found.sort(key=lambda x: x['start'])
        
    if not found: return bot.send_message(message.chat.id, "Пар нет!")
    
    text = f"📅 **{target_date.strftime('%d.%m')}**\n\n"
    for l in found:
        text += f"⏰ {l['start'].strftime('%H:%M')}\n📚 {l['subject']}\n🚪 {l['room']}\n"
        if user['role'] == 'teacher': text += f"👥 {l['group']}\n"
        else: text += f"👨‍🏫 {l['teacher']}\n"
        text += "➖\n"
    bot.send_message(message.chat.id, text, parse_mode='Markdown')

@bot.message_handler(func=lambda m: m.text == "🔙 Назад")
def back(m): start(m)

@bot.callback_query_handler(func=lambda call: call.data.startswith('here_'))
def checkin(call):
    lid = call.data.split('here_')[1]
    if att_mgr.process_student_click(lid, call.message.chat.id):
        bot.edit_message_text("✅ Вы отмечены! Отдыхайте.", call.message.chat.id, call.message.message_id)
    else:
        bot.answer_callback_query(call.id, "Ошибка")

if __name__ == "__main__":
    bot.infinity_polling()

#  Проблема Памяти и Диска 

Вы загрузили новый файл через наш красивый сайт.
Но бот продолжает слать старое расписание. Почему?

Это классическая проблема архитектуры:
* **Disk (SSD):** Файл обновился.
* **RAM (Память):** Бот загрузил данные в переменную `data_mgr.schedule` один раз, когда вы нажали Run.

Бот не умеет читать мысли. Ему нужно **явно** сказать: "Эй, данные на диске изменились, перечитай их!".

### Решение: Команда `/reload`

Мы добавим специальную команду для администратора.
Алгоритм:
1. Админ пишет `/reload`.
2. Бот вызывает метод `data_mgr.load_all()`.
3. Парсер заново читает Excel.
4. Переменные в памяти обновляются.
"""

In [ ]:
import telebot
from telebot import types
import threading
import os
from datetime import datetime, timedelta
from dotenv import load_dotenv

# Импортируем наши классы (убедитесь, что файлы лежат рядом или ячейки с ними запущены)
from data_manager import DataManager
from attendance_manager import AttendanceManager

# --- ИНИЦИАЛИЗАЦИЯ ---
load_dotenv()
TOKEN = os.getenv('BOT_TOKEN') 
if not TOKEN: raise ValueError("❌ Ошибка: Нет токена в .env файле!")

TEACHER_PASSWORD = "12345" # Пароль для регистрации преподавателей

bot = telebot.TeleBot(TOKEN)

# Логика выбора файла:
# Если админ загрузил новый файл через сайт - берем его.
# Если нет - берем старый (дефолтный).
schedule_path = 'data/uploaded_schedule.xlsx'
if not os.path.exists(schedule_path):
    print(f"⚠️ Файл {schedule_path} не найден. Использую резервный файл.")
    schedule_path = 'МШИБС_расписание_2025-2026_1сем_14нед_v1.xlsx'

# 1. Запускаем Менеджеров
data_mgr = DataManager(schedule_path, 'users.json')
att_mgr = AttendanceManager(data_mgr, bot)

# 2. Запускаем Планировщик в фоне
threading.Thread(target=att_mgr.run_scheduler_loop, daemon=True).start()
print("🚀 Бот запущен! Система готова к работе.")

# ==========================================
# 🔧 АДМИНИСТРАТИВНЫЕ КОМАНДЫ
# ==========================================

@bot.message_handler(commands=['reload'])
def admin_reload(message):
    """
    Горячая перезагрузка.
    Позволяет обновить расписание в памяти бота без перезапуска скрипта.
    Используется после загрузки файла через веб-сайт.
    """
    bot.send_message(message.chat.id, "🔄 Обновляю данные с диска...")
    try:
        data_mgr.load_all()
        # Статистика для админа
        stats = (f"✅ Данные обновлены!\n"
                 f"📂 Файл: {data_mgr.schedule_file}\n"
                 f"📚 Групп: {len(data_mgr.schedule)}\n"
                 f"👨‍🏫 Преподавателей: {len(data_mgr.teachers_list)}")
        bot.send_message(message.chat.id, stats)
        print(f"System reloaded by {message.from_user.username}")
    except Exception as e:
        bot.send_message(message.chat.id, f"❌ Ошибка при обновлении: {e}")

@bot.message_handler(commands=['notify'])
def admin_notify(message):
    """
    Принудительная рассылка напоминаний.
    Использование: /notify 2 (напомнить о парах, которые начнутся через 2 часа)
    """
    try:
        parts = message.text.split()
        hours = float(parts[1]) if len(parts) > 1 else 1.0
        
        bot.send_message(message.chat.id, f"📣 Ищу пары, которые начнутся через {hours} ч...")
        count = att_mgr.manual_trigger(hours)
        bot.reply_to(message, f"✅ Рассылка завершена. Уведомлено групп: {count}")
    except ValueError:
        bot.reply_to(message, "❌ Ошибка формата. Пишите: `/notify 2`")

@bot.message_handler(commands=['test'])
def admin_test_simulation(message):
    """
    Запускает симуляцию лекции для демонстрации.
    Создает фейковую пару 'через час' и присылает уведомления.
    """
    bot.send_message(message.chat.id, "🚀 Запускаю симуляцию лекции...")
    # Запускаем тест для конкретного преподавателя (зарегистрируйтесь под этим именем!)
    result = att_mgr.run_test_scenario("Воробьев Даниил Анатольевич")
    bot.send_message(message.chat.id, result)

# ==========================================
# 🎓 ПОЛЬЗОВАТЕЛЬСКИЙ ИНТЕРФЕЙС (Логика меню)
# ==========================================

@bot.message_handler(commands=['start'])
def start_handler(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("🎓 Студент", "👨‍🏫 Преподаватель")
    bot.send_message(message.chat.id, "👋 Привет! Добро пожаловать в Бот-Старосту.\nВыберите вашу роль:", reply_markup=markup)

# --- ЛОГИКА СТУДЕНТА ---
@bot.message_handler(func=lambda m: m.text == "🎓 Студент")
def register_student_step1(message):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    # Генерируем кнопки групп из памяти
    groups = list(data_mgr.schedule.keys())
    for group in groups:
        markup.add(types.KeyboardButton(group))
    markup.row("🔙 Назад")
    bot.send_message(message.chat.id, "Выберите вашу учебную группу:", reply_markup=markup)

@bot.message_handler(func=lambda m: m.text in data_mgr.schedule.keys())
def register_student_finish(message):
    data_mgr.save_user(message.chat.id, "student", message.text)
    send_main_menu(message.chat.id, f"✅ Вы успешно записаны в группу **{message.text}**!")

# --- ЛОГИКА ПРЕПОДАВАТЕЛЯ ---
@bot.message_handler(func=lambda m: m.text == "👨‍🏫 Преподаватель")
def register_teacher_step1(message):
    msg = bot.send_message(message.chat.id, "🔒 Введите код доступа преподавателя:", reply_markup=types.ReplyKeyboardRemove())
    bot.register_next_step_handler(msg, verify_teacher_password)

def verify_teacher_password(message):
    if message.text == TEACHER_PASSWORD:
        markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
        # Показываем список преподавателей (первые 30, чтобы не перегружать экран)
        for t in data_mgr.teachers_list[:30]: 
            markup.add(types.KeyboardButton(t))
        bot.send_message(message.chat.id, "🔓 Доступ разрешен. Выберите ваше ФИО:", reply_markup=markup)
        bot.register_next_step_handler(message, register_teacher_finish)
    else:
        bot.send_message(message.chat.id, "⛔ Неверный пароль. Доступ запрещен.")
        start_handler(message)

def register_teacher_finish(message):
    if message.text in data_mgr.teachers_list:
        data_mgr.save_user(message.chat.id, "teacher", message.text)
        send_main_menu(message.chat.id, f"✅ Здравствуйте, **{message.text}**! Теперь вы будете получать отчеты.")
    else:
        bot.send_message(message.chat.id, "⚠️ Такого преподавателя нет в расписании. Попробуйте снова.")
        start_handler(message)

# --- ОБЩЕЕ МЕНЮ И ВЫВОД РАСПИСАНИЯ ---
def send_main_menu(chat_id, text):
    markup = types.ReplyKeyboardMarkup(resize_keyboard=True)
    markup.row("📅 На сегодня", "📅 На завтра")
    markup.row("🔙 Сменить роль")
    bot.send_message(chat_id, text, reply_markup=markup, parse_mode='Markdown')

@bot.message_handler(func=lambda m: m.text in ["📅 На сегодня", "📅 На завтра"])
def show_schedule(message):
    # 1. Проверяем регистрацию
    user = data_mgr.users.get(str(message.chat.id))
    if not user:
        bot.send_message(message.chat.id, "⚠️ Сначала зарегистрируйтесь через /start")
        return

    # 2. Определяем дату
    target_date = datetime.now().date()
    if "На завтра" in message.text:
        target_date += timedelta(days=1)

    # 3. Ищем пары (Алгоритм зависит от роли)
    found_lessons = []
    role = user['role']
    target_name = user['info'] # Группа или ФИО

    if role == 'student':
        # Студент: ищем по ключу группы
        lessons = data_mgr.schedule.get(target_name, [])
        found_lessons = [l for l in lessons if l['start'].date() == target_date]
    
    elif role == 'teacher':
        # Преподаватель: ищем по всей базе
        for group_name, lessons in data_mgr.schedule.items():
            for l in lessons:
                if l['start'].date() == target_date and l['teacher'] == target_name:
                    l_copy = l.copy()
                    l_copy['group_name'] = group_name # Добавляем группу, чтобы препод знал, к кому идти
                    found_lessons.append(l_copy)
        found_lessons.sort(key=lambda x: x['start'])

    # 4. Вывод
    if not found_lessons:
        bot.send_message(message.chat.id, f"🎉 На {target_date.strftime('%d.%m')} пар нет! Отдыхайте.")
        return

    text = f"📅 **Расписание на {target_date.strftime('%d.%m')}**\n\n"
    for l in found_lessons:
        text += f"⏰ `{l['start'].strftime('%H:%M')} - {l['end'].strftime('%H:%M')}`\n"
        text += f"📚 {l['subject']}\n"
        text += f"🚪 {l['room']}\n"
        
        if role == 'teacher':
            text += f"👥 Группа: **{l['group_name']}**\n"
        else:
            text += f"👨‍🏫 {l['teacher']}\n"
        text += "➖➖➖➖➖➖➖\n"

    bot.send_message(message.chat.id, text, parse_mode='Markdown')

# --- КНОПКА НАЗАД ---
@bot.message_handler(func=lambda m: m.text in ["🔙 Назад", "🔙 Сменить роль"])
def go_back(message):
    start_handler(message)

# --- ОБРАБОТКА ИНЛАЙН-КНОПКИ (CHECK-IN) ---
@bot.callback_query_handler(func=lambda call: call.data.startswith('here_'))
def handle_checkin_button(call):
    # Достаем ID урока из кнопки
    lesson_id = call.data.split('here_')[1]
    
    # Делегируем логику менеджеру посещаемости
    success = att_mgr.process_student_click(lesson_id, call.message.chat.id)
    
    if success:
        bot.answer_callback_query(call.id, "Отлично! Вы отмечены ✅")
        bot.edit_message_text(
            f"✅ {datetime.now().strftime('%H:%M')} Присутствие подтверждено.", 
            call.message.chat.id, 
            call.message.message_id
        )
    else:
        bot.answer_callback_query(call.id, "⛔ Ошибка: время вышло или вы уже отметились.")

# --- ЗАПУСК ---
if __name__ == "__main__":
    bot.infinity_polling()